<a href="https://colab.research.google.com/github/Ricardo-web-ctrl/CryptoBot_AI_v1.ipynb/blob/main/CryptoBot_AI_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# CRYPTOBOT AI - INSTALACIÓN DE HERRAMIENTAS
# Ejecuta esta celda una sola vez al iniciar el entorno.

!pip -q install pandas numpy requests matplotlib

print("================================")
print("     CRYPTOBOT AI v1.0")
print("================================")
print("Herramientas instaladas correctamente.")
print("Entorno: Google Colab")
print("Modo: DEMO / SIMULACIÓN")


     CRYPTOBOT AI v1.0
Herramientas instaladas correctamente.
Entorno: Google Colab
Modo: DEMO / SIMULACIÓN


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# ==========================================
# CRYPTOBOT AI - MÓDULO 1
# CONEXIÓN CON BINANCE Y ANÁLISIS DE BTC
# ==========================================

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timezone

# CONFIGURACIÓN
BASE_URL = "https://data-api.binance.vision"
SYMBOL = "BTCUSDT"
TIMEFRAME = "15m"

print("Iniciando CryptoBot AI...")

# 1. OBTENER VELAS DE BINANCE
url = f"{BASE_URL}/api/v3/klines"

params = {
    "symbol": SYMBOL,
    "interval": TIMEFRAME,
    "limit": 200
}

response = requests.get(url, params=params, timeout=20)
response.raise_for_status()

data = response.json()

# 2. ORGANIZAR LOS DATOS
columns = [
    "time", "open", "high", "low", "close",
    "volume", "close_time", "quote_volume",
    "trades", "taker_buy", "taker_quote", "ignore"
]

df = pd.DataFrame(data, columns=columns)

# Convertir precios y volumen a números
for col in ["open", "high", "low", "close", "volume"]:
    df[col] = pd.to_numeric(df[col])

df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)

# Excluir la vela que todavía no ha cerrado
df = df.iloc[:-1].copy()

# 3. CALCULAR MEDIAS MÓVILES
df["EMA21"] = df["close"].ewm(
    span=21, adjust=False
).mean()

df["EMA50"] = df["close"].ewm(
    span=50, adjust=False
).mean()

# 4. CALCULAR RSI 14
delta = df["close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.ewm(
    alpha=1/14, adjust=False, min_periods=14
).mean()

avg_loss = loss.ewm(
    alpha=1/14, adjust=False, min_periods=14
).mean()

rs = avg_gain / avg_loss.replace(0, np.nan)
df["RSI14"] = 100 - (100 / (1 + rs))

# 5. ANALIZAR LA ÚLTIMA VELA CERRADA
last = df.iloc[-1]

price = last["close"]
ema21 = last["EMA21"]
ema50 = last["EMA50"]
rsi = last["RSI14"]

# 6. GENERAR SEÑAL INICIAL
if ema21 > ema50 and 50 < rsi < 70:
    signal = "POSIBLE LONG"

elif ema21 < ema50 and 30 < rsi < 50:
    signal = "POSIBLE SHORT"

else:
    signal = "ESPERAR"

# 7. MOSTRAR RESULTADOS
print("\n========== RESULTADOS ==========")

print("Mercado:", SYMBOL)
print("Temporalidad:", TIMEFRAME)
print("Última vela cerrada:", last["time"])
print("Precio BTC:", round(price, 2))
print("EMA 21:", round(ema21, 2))
print("EMA 50:", round(ema50, 2))
print("RSI 14:", round(rsi, 2))

print("\nSEÑAL DEL ALGORITMO:", signal)

print("\nModo: SIMULACIÓN")
print("Órdenes reales: DESACTIVADAS")
print("Saldo utilizado: 0 USDT")

print("\nCryptoBot AI ejecutado correctamente.")


Iniciando CryptoBot AI...

========== RESULTADOS ==========
Mercado: BTCUSDT
Temporalidad: 15m
Última vela cerrada: 2026-11-25 01:45:00+00:00
Precio BTC: 59272.01
EMA 21: 59686.3
EMA 50: 59860.88
RSI 14: 33.09

SEÑAL DEL ALGORITMO: POSIBLE SHORT

Modo: SIMULACIÓN
Órdenes reales: DESACTIVADAS
Saldo utilizado: 0 USDT

CryptoBot AI ejecutado correctamente.


In [ ]:

# ==================================================
# CRYPTOBOT AI - MODULO 3.1
# DESCARGA DE DATOS HISTORICOS BINANCE
# BABYUSDT FUTURES
# ==================================================

import requests
import pandas as pd
import zipfile
from io import BytesIO

SIMBOLO = "BABYUSDT"
FECHA = "2026-11-23"

TEMPORALIDADES = ["4h", "1h", "15m", "5m"]

BASE_URL = (
    "https://data.binance.vision/data/futures/um/"
    "daily/klines"
)

COLUMNAS = [
    "open_time", "open", "high", "low", "close",
    "volume", "close_time", "quote_asset_volume",
    "number_of_trades", "taker_buy_base_asset_volume",
    "taker_buy_quote_asset_volume", "ignore"
]


def descargar_velas(intervalo):

    nombre = f"{SIMBOLO}-{intervalo}-{FECHA}.zip"

    url = (
        f"{BASE_URL}/{SIMBOLO}/"
        f"{intervalo}/{nombre}"
    )

    respuesta = requests.get(url, timeout=60)
    respuesta.raise_for_status()

    with zipfile.ZipFile(BytesIO(respuesta.content)) as archivo:

        csv_nombre = next(
            (f for f in archivo.namelist()
             if f.endswith(".csv")),
            None
        )

        if csv_nombre is None:
            raise ValueError("El ZIP no contiene CSV")

        with archivo.open(csv_nombre) as csv:

            df = pd.read_csv(
                csv,
                header=0,
                names=COLUMNAS
            )

    # Eliminar encabezados repetidos o filas incorrectas
    df = df[
        pd.to_numeric(
            df["open_time"],
            errors="coerce"
        ).notna()
    ].copy()

    # Convertir fechas
    df["open_time"] = pd.to_datetime(
        pd.to_numeric(df["open_time"]),
        unit="ms",
        utc=True
    )

    df["close_time"] = pd.to_datetime(
        pd.to_numeric(df["close_time"]),
        unit="ms",
        utc=True
    )

    # Convertir datos numericos
    numericas = [
        "open", "high", "low", "close", "volume",
        "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume"
    ]

    for columna in numericas:
        df[columna] = pd.to_numeric(
            df[columna],
            errors="coerce"
        )

    # Limpiar datos
    df = df.dropna(
        subset=["open", "high", "low", "close", "volume"]
    )

    df = (
        df.drop_duplicates(subset=["open_time"])
          .sort_values("open_time")
          .reset_index(drop=True)
    )

    return df


# DESCARGAR TEMPORALIDADES
datos = {}

for intervalo in TEMPORALIDADES:

    print(f"\nDescargando BABYUSDT - {intervalo}...")

    datos[intervalo] = descargar_velas(intervalo)

    print("Velas:", len(datos[intervalo]))
    print("Ultima fecha:", datos[intervalo]["open_time"].iloc[-1])
    print("Ultimo cierre:", datos[intervalo]["close"].iloc[-1])


# VARIABLES PARA LOS SIGUIENTES MODULOS
baby_4h = datos["4h"]
baby_1h = datos["1h"]
baby_15m = datos["15m"]
baby_5m = datos["5m"]


# RESULTADO
print("\n======================================")
print("CRYPTOBOT AI - MODULO 3 COMPLETADO")
print("======================================")

for intervalo, df in datos.items():
    print(intervalo, ":", len(df), "velas")

print("\nULTIMOS PRECIOS:")
print("4H:", baby_4h["close"].iloc[-1])
print("1H:", baby_1h["close"].iloc[-1])
print("15M:", baby_15m["close"].iloc[-1])
print("5M:", baby_5m["close"].iloc[-1])

print("\nMODULO 3.1 LISTO")



Descargando BABYUSDT - 4h...
Velas: 6
Ultima fecha: 2026-11-23 20:00:00+00:00
Ultimo cierre: 0.008481

Descargando BABYUSDT - 1h...
Velas: 24
Ultima fecha: 2026-11-23 23:00:00+00:00
Ultimo cierre: 0.008481

Descargando BABYUSDT - 15m...
Velas: 96
Ultima fecha: 2026-11-23 23:45:00+00:00
Ultimo cierre: 0.008481

Descargando BABYUSDT - 5m...
Velas: 288
Ultima fecha: 2026-11-23 23:55:00+00:00
Ultimo cierre: 0.008481

CRYPTOBOT AI - MODULO 3 COMPLETADO
4h : 6 velas
1h : 24 velas
15m : 96 velas
5m : 288 velas

ULTIMOS PRECIOS:
4H: 0.008481
1H: 0.008481
15M: 0.008481
5M: 0.008481

MODULO 3.1 LISTO


In [ ]:

# =====================================================
# CRYPTOBOT AI - MODULO 3.2
# HISTORIAL DE 30 DIAS - BABYUSDT FUTURES
# FUENTE: BINANCE PUBLIC DATA
# TEMPORALIDADES: 4H / 1H / 15M / 5M
# =====================================================

import requests
import pandas as pd
import zipfile
import time

from io import BytesIO
from datetime import date, timedelta

# 1. CONFIGURACION
SIMBOLO = "BABYUSDT"

FECHA_FINAL = date(2026, 11, 23)
DIAS_HISTORIAL = 30

FECHA_INICIAL = FECHA_FINAL - timedelta(
    days=DIAS_HISTORIAL - 1
)

TEMPORALIDADES = ["4h", "1h", "15m", "5m"]

BASE_URL = (
    "https://data.binance.vision/data/futures/um/"
    "daily/klines"
)

COLUMNAS = [
    "open_time", "open", "high", "low", "close",
    "volume", "close_time", "quote_asset_volume",
    "number_of_trades", "taker_buy_base_asset_volume",
    "taker_buy_quote_asset_volume", "ignore"
]

sesion = requests.Session()

# 2. FUNCION PARA DESCARGAR UN DIA
def descargar_dia(intervalo, fecha):

    fecha_texto = fecha.strftime("%Y-%m-%d")

    nombre = (
        f"{SIMBOLO}-{intervalo}-{fecha_texto}.zip"
    )

    url = (
        f"{BASE_URL}/{SIMBOLO}/"
        f"{intervalo}/{nombre}"
    )

    for intento in range(3):

        respuesta = sesion.get(url, timeout=45)

        if respuesta.status_code == 404:
            return None

        if respuesta.status_code == 200:
            break

        if respuesta.status_code in [429, 500, 502, 503, 504]:
            time.sleep(2 * (intento + 1))
            continue

        respuesta.raise_for_status()

    else:
        respuesta.raise_for_status()

    with zipfile.ZipFile(
        BytesIO(respuesta.content)
    ) as archivo:

        csv_nombre = next(
            (f for f in archivo.namelist()
             if f.endswith(".csv")),
            None
        )

        if csv_nombre is None:
            raise ValueError(
                f"CSV no encontrado: {nombre}"
            )

        with archivo.open(csv_nombre) as csv:
            df = pd.read_csv(
                csv,
                header=None,
                names=COLUMNAS
            )

    # 3. LIMPIAR ENCABEZADOS
    df["open_time"] = pd.to_numeric(
        df["open_time"],
        errors="coerce"
    )

    df["close_time"] = pd.to_numeric(
        df["close_time"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["open_time", "close_time"]
    ).copy()

    # 4. CONVERTIR FECHAS
    df["open_time"] = pd.to_datetime(
        df["open_time"],
        unit="ms",
        utc=True
    )

    df["close_time"] = pd.to_datetime(
        df["close_time"],
        unit="ms",
        utc=True
    )

    # 5. CONVERTIR PRECIOS Y VOLUMEN
    numericas = [
        "open", "high", "low", "close", "volume",
        "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume"
    ]

    for columna in numericas:
        df[columna] = pd.to_numeric(
            df[columna],
            errors="coerce"
        )

    df = df.dropna(
        subset=["open", "high", "low", "close", "volume"]
    )

    return df


# 6. DESCARGAR LOS 30 DIAS
datos = {}
fechas_faltantes = {}

for intervalo in TEMPORALIDADES:

    print("\n================================")
    print("DESCARGANDO:", intervalo)
    print("================================")

    partes = []
    faltantes = []

    fecha_actual = FECHA_INICIAL

    while fecha_actual <= FECHA_FINAL:

        print(
            "Fecha:",
            fecha_actual.strftime("%Y-%m-%d")
        )

        try:
            df_dia = descargar_dia(
                intervalo,
                fecha_actual
            )

            if df_dia is None:
                faltantes.append(
                    fecha_actual.strftime("%Y-%m-%d")
                )
            else:
                partes.append(df_dia)

        except Exception as error:
            print("ERROR:", str(error))
            raise

        fecha_actual += timedelta(days=1)
        time.sleep(0.1)

    if not partes:
        raise ValueError(
            f"No se descargaron datos para {intervalo}"
        )

    df_final = pd.concat(
        partes,
        ignore_index=True
    )

    df_final = (
        df_final
        .drop_duplicates(subset=["open_time"])
        .sort_values("open_time")
        .reset_index(drop=True)
    )

    datos[intervalo] = df_final
    fechas_faltantes[intervalo] = faltantes

    print("Velas acumuladas:", len(df_final))
    print("Dias sin archivo:", len(faltantes))


# 7. CREAR VARIABLES PARA LOS SIGUIENTES MODULOS
baby_4h = datos["4h"]
baby_1h = datos["1h"]
baby_15m = datos["15m"]
baby_5m = datos["5m"]


# 8. INFORME FINAL
print("\n========================================")
print("CRYPTOBOT AI - MODULO 3.2 COMPLETADO")
print("========================================")

print("Par:", SIMBOLO)
print("Desde:", FECHA_INICIAL)
print("Hasta:", FECHA_FINAL)

for intervalo, df in datos.items():

    print("\nTemporalidad:", intervalo)
    print("Total de velas:", len(df))
    print("Primera vela:", df["open_time"].iloc[0])
    print("Ultima vela:", df["open_time"].iloc[-1])
    print("Ultimo cierre:", df["close"].iloc[-1])

print("\nVARIABLES CREADAS:")
print("baby_4h")
print("baby_1h")
print("baby_15m")
print("baby_5m")

print("\nMODULO 3.2 LISTO")



DESCARGANDO: 4h
Fecha: 2026-10-25
Fecha: 2026-10-26
Fecha: 2026-10-27
Fecha: 2026-10-28
Fecha: 2026-10-29
Fecha: 2026-10-30
Fecha: 2026-10-31
Fecha: 2026-11-01
Fecha: 2026-11-02
Fecha: 2026-11-03
Fecha: 2026-11-04
Fecha: 2026-11-05
Fecha: 2026-11-06
Fecha: 2026-11-07
Fecha: 2026-11-08
Fecha: 2026-11-09
Fecha: 2026-11-10
Fecha: 2026-11-11
Fecha: 2026-11-12
Fecha: 2026-11-13
Fecha: 2026-11-14
Fecha: 2026-11-15
Fecha: 2026-11-16
Fecha: 2026-11-17
Fecha: 2026-11-18
Fecha: 2026-11-19
Fecha: 2026-11-20
Fecha: 2026-11-21
Fecha: 2026-11-22
Fecha: 2026-11-23
Velas acumuladas: 180
Dias sin archivo: 0

DESCARGANDO: 1h
Fecha: 2026-10-25
Fecha: 2026-10-26
Fecha: 2026-10-27
Fecha: 2026-10-28
Fecha: 2026-10-29
Fecha: 2026-10-30
Fecha: 2026-10-31
Fecha: 2026-11-01
Fecha: 2026-11-02
Fecha: 2026-11-03
Fecha: 2026-11-04
Fecha: 2026-11-05
Fecha: 2026-11-06
Fecha: 2026-11-07
Fecha: 2026-11-08
Fecha: 2026-11-09
Fecha: 2026-11-10
Fecha: 2026-11-11
Fecha: 2026-11-12
Fecha: 2026-11-13
Fecha: 2026-11-14
Fecha:

In [ ]:

# ============================================================
# CRYPTOBOT AI — BABYUSDT
# MODULO 3.3 — DIAGNOSTICO DE DATOS EN VIVO
# Solo consulta publica. No utiliza API keys ni opera.
# ============================================================

import requests
import pandas as pd
from datetime import datetime, timezone

SIMBOLO = "BABYUSDT"
INTERVALO = "1m"

URL = "https://fapi.binance.com/fapi/v1/klines"

print("====================================")
print("CRYPTOBOT AI — CONEXION EN VIVO")
print("====================================")
print("Simbolo:", SIMBOLO)
print("Intervalo:", INTERVALO)
print("Modo: FUTURES DEMO / SOLO LECTURA")

try:

    respuesta = requests.get(
        URL,
        params={
            "symbol": SIMBOLO,
            "interval": INTERVALO,
            "limit": 5
        },
        timeout=15
    )

    print("\nHTTP:", respuesta.status_code)

    if respuesta.status_code == 200:

        datos = respuesta.json()

        columnas = [
            "open_time", "open", "high", "low",
            "close", "volume", "close_time",
            "quote_volume", "trades",
            "taker_buy_base", "taker_buy_quote",
            "ignore"
        ]

        df_vivo = pd.DataFrame(
            datos,
            columns=columnas
        )

        df_vivo["open_time"] = pd.to_datetime(
            df_vivo["open_time"],
            unit="ms",
            utc=True
        )

        df_vivo["close_time"] = pd.to_datetime(
            df_vivo["close_time"],
            unit="ms",
            utc=True
        )

        for columna in [
            "open", "high", "low", "close", "volume"
        ]:
            df_vivo[columna] = pd.to_numeric(
                df_vivo[columna]
            )

        ahora = pd.Timestamp.now(tz="UTC")

        # Solo velas completamente cerradas
        df_cerrado = df_vivo[
            df_vivo["close_time"] < ahora
        ].copy()

        print("\nCONEXION EXITOSA")
        print("Velas recibidas:", len(df_vivo))
        print("Velas cerradas:", len(df_cerrado))

        if not df_cerrado.empty:

            ultima = df_cerrado.iloc[-1]

            print("\nULTIMA VELA CERRADA")
            print("Fecha UTC:", ultima["close_time"])
            print("Cierre:", ultima["close"])
            print("Volumen:", ultima["volume"])

            antiguedad = (
                ahora - ultima["close_time"]
            ).total_seconds()

            print("Antiguedad en segundos:", round(antiguedad, 1))

            if antiguedad <= 180:
                print("\nESTADO: DATOS RECIENTES")
            else:
                print("\nESTADO: DATOS DEMASIADO ANTIGUOS")

        else:
            print("No hay velas cerradas disponibles.")

    elif respuesta.status_code in [403, 451]:

        print("\nACCESO RESTRINGIDO")
        print("Binance no permite esta consulta desde")
        print("el entorno de ejecucion actual.")
        print("No se intentara evadir la restriccion.")

    else:

        print("\nRESPUESTA NO ESPERADA")
        print(respuesta.text[:500])

except requests.exceptions.Timeout:

    print("\nERROR: tiempo de espera agotado.")

except requests.exceptions.RequestException as error:

    print("\nERROR DE CONEXION:")
    print(str(error))

except Exception as error:

    print("\nERROR:")
    print(str(error))

print("\n====================================")
print("MODULO 3.3 FINALIZADO")
print("No se enviaron ordenes.")
print("====================================")


CRYPTOBOT AI — CONEXION EN VIVO
Simbolo: BABYUSDT
Intervalo: 1m
Modo: FUTURES DEMO / SOLO LECTURA

HTTP: 451

ACCESO RESTRINGIDO
Binance no permite esta consulta desde
el entorno de ejecucion actual.
No se intentara evadir la restriccion.

MODULO 3.3 FINALIZADO
No se enviaron ordenes.


In [ ]:

# ============================================================
# CRYPTOBOT AI — BABYUSDT
# MODULO 3.4 — CONEXION BYBIT / SOLO LECTURA
# No utiliza API keys. No ejecuta operaciones.
# ============================================================

import requests
import pandas as pd
from datetime import datetime, timezone

SIMBOLO = "BABYUSDT"
BASE_URL = "https://api.bybit.com"

print("========================================")
print("CRYPTOBOT AI — MODULO 3.4")
print("PROVEEDOR SECUNDARIO: BYBIT")
print("========================================")

sesion = requests.Session()
sesion.headers.update({
    "User-Agent": "CryptoBot-AI-ReadOnly/1.0"
})

try:

    # --------------------------------------------------------
    # 1. COMPROBAR SI EXISTE BABYUSDT FUTURES LINEAR
    # --------------------------------------------------------

    print("\n1. VERIFICANDO CONTRATO...")

    url_instrumento = (
        BASE_URL + "/v5/market/instruments-info"
    )

    respuesta = sesion.get(
        url_instrumento,
        params={
            "category": "linear",
            "symbol": SIMBOLO
        },
        timeout=15
    )

    print("HTTP instrumentos:", respuesta.status_code)
    respuesta.raise_for_status()

    resultado = respuesta.json()

    if resultado.get("retCode") != 0:
        raise Exception(
            "Bybit: " + resultado.get("retMsg", "Error desconocido")
        )

    lista = resultado.get("result", {}).get("list", [])

    if not lista:
        raise Exception(
            "BABYUSDT no aparece como contrato linear en Bybit."
        )

    instrumento = lista[0]

    estado = instrumento.get("status")
    tipo = instrumento.get("contractType")

    print("Símbolo:", instrumento.get("symbol"))
    print("Estado:", estado)
    print("Tipo de contrato:", tipo)

    if estado != "Trading":
        raise Exception(
            "El contrato no está habilitado para trading actualmente."
        )

    # --------------------------------------------------------
    # 2. DESCARGAR VELAS DE UN MINUTO
    # --------------------------------------------------------

    print("\n2. DESCARGANDO VELAS...")

    url_kline = BASE_URL + "/v5/market/kline"

    respuesta = sesion.get(
        url_kline,
        params={
            "category": "linear",
            "symbol": SIMBOLO,
            "interval": "1",
            "limit": 200
        },
        timeout=15
    )

    print("HTTP velas:", respuesta.status_code)
    respuesta.raise_for_status()

    resultado = respuesta.json()

    if resultado.get("retCode") != 0:
        raise Exception(
            "Bybit: " + resultado.get("retMsg", "Error en velas")
        )

    velas = resultado.get("result", {}).get("list", [])

    if not velas:
        raise Exception("Bybit no devolvió velas.")

    columnas = [
        "open_time", "open", "high", "low",
        "close", "volume", "turnover"
    ]

    bybit_1m = pd.DataFrame(
        velas,
        columns=columnas
    )

    # Bybit entrega las velas en orden descendente.
    bybit_1m["open_time"] = pd.to_datetime(
        pd.to_numeric(bybit_1m["open_time"]),
        unit="ms",
        utc=True
    )

    for columna in [
        "open", "high", "low", "close",
        "volume", "turnover"
    ]:
        bybit_1m[columna] = pd.to_numeric(
            bybit_1m[columna]
        )

    bybit_1m = (
        bybit_1m
        .sort_values("open_time")
        .drop_duplicates("open_time")
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 3. SEPARAR VELAS CERRADAS DE LA VELA EN FORMACION
    # --------------------------------------------------------

    ahora = pd.Timestamp.now(tz="UTC")

    bybit_1m["close_time"] = (
        bybit_1m["open_time"]
        + pd.Timedelta(minutes=1)
    )

    bybit_1m_cerrado = bybit_1m[
        bybit_1m["close_time"] <= ahora
    ].copy()

    if bybit_1m_cerrado.empty:
        raise Exception("Todavía no hay velas cerradas.")

    ultima = bybit_1m_cerrado.iloc[-1]

    # --------------------------------------------------------
    # 4. RESULTADO
    # --------------------------------------------------------

    print("\n========================================")
    print("CONEXION BYBIT VALIDADA")
    print("========================================")

    print("Fuente: BYBIT")
    print("Mercado: FUTURES LINEAR")
    print("Símbolo:", SIMBOLO)
    print("Velas recibidas:", len(bybit_1m))
    print("Velas cerradas:", len(bybit_1m_cerrado))

    print("\nULTIMA VELA CERRADA")
    print("Hora UTC:", ultima["close_time"])
    print("Open:", ultima["open"])
    print("High:", ultima["high"])
    print("Low:", ultima["low"])
    print("Close:", ultima["close"])
    print("Volume:", ultima["volume"])

    antiguedad = (
        ahora - ultima["close_time"]
    ).total_seconds()

    print("\nAntigüedad de la última vela:", round(antiguedad, 1), "segundos")

    if antiguedad <= 180:
        print("Estado de datos: RECIENTES")
    else:
        print("Estado de datos: ANTIGUOS / REVISAR")

    print("\n========================================")
    print("MODULO 3.4 FINALIZADO")
    print("SOLO LECTURA — SIN ORDENES")
    print("========================================")

except requests.exceptions.RequestException as error:

    print("\nERROR DE CONEXION:")
    print(str(error))

except Exception as error:

    print("\nNO SE PUDO VALIDAR BYBIT:")
    print(str(error))

finally:
    sesion.close()


CRYPTOBOT AI — MODULO 3.4
PROVEEDOR SECUNDARIO: BYBIT

1. VERIFICANDO CONTRATO...
HTTP instrumentos: 403

ERROR DE CONEXION:
403 Client Error: Forbidden for url: https://api.bybit.com/v5/market/instruments-info?category=linear&symbol=BABYUSDT


In [ ]:

# ============================================================
# CRYPTOBOT AI — MODULO 3.4.1
# DIAGNOSTICO DE CONEXION BYBIT
# SOLO LECTURA — SIN ORDENES
# ============================================================

import requests

URL = "https://api.bytick.com/v5/market/instruments-info"

print("CRYPTOBOT AI — DIAGNOSTICO BYBIT")
print("Probando endpoint alternativo oficial...")

try:
    respuesta = requests.get(
        URL,
        params={
            "category": "linear",
            "symbol": "BABYUSDT"
        },
        timeout=15
    )

    print("HTTP:", respuesta.status_code)

    if respuesta.status_code == 200:
        datos = respuesta.json()

        print("Respuesta API:", datos.get("retMsg"))

        contratos = datos.get("result", {}).get("list", [])

        if contratos:
            print("CONTRATO ENCONTRADO:")
            print(contratos[0].get("symbol"))
            print("Estado:", contratos[0].get("status"))
        else:
            print("No se encontró BABYUSDT.")

    elif respuesta.status_code == 403:
        print("ACCESO DENEGADO POR EL SERVIDOR.")

    else:
        print(respuesta.text[:500])

except Exception as error:
    print("ERROR:", str(error))

print("DIAGNOSTICO TERMINADO")
print("No se enviaron órdenes.")


CRYPTOBOT AI — DIAGNOSTICO BYBIT
Probando endpoint alternativo oficial...
HTTP: 403
ACCESO DENEGADO POR EL SERVIDOR.
DIAGNOSTICO TERMINADO
No se enviaron órdenes.


In [ ]:

# ============================================================
# CRYPTOBOT AI — MODULO 3.4.2
# PROVEEDOR DE DATOS: OKX
# SOLO LECTURA — SIN ORDENES
# ============================================================

import requests
import pandas as pd

BASE_URL = "https://www.okx.com"
SIMBOLO = "BABY-USDT-SWAP"

print("======================================")
print("CRYPTOBOT AI — DIAGNOSTICO OKX")
print("======================================")

sesion = requests.Session()
sesion.headers.update({
    "User-Agent": "CryptoBot-AI-ReadOnly/1.0"
})

try:

    # 1. VERIFICAR CONTRATO PERPETUO

    print("\n1. VERIFICANDO CONTRATO...")

    r = sesion.get(
        BASE_URL + "/api/v5/public/instruments",
        params={
            "instType": "SWAP",
            "instId": SIMBOLO
        },
        timeout=15
    )

    print("HTTP instrumentos:", r.status_code)
    r.raise_for_status()

    datos = r.json()

    if datos.get("code") != "0":
        raise Exception(datos.get("msg", "Error de OKX"))

    contratos = datos.get("data", [])

    if not contratos:
        raise Exception(
            "OKX no encontró el contrato " + SIMBOLO
        )

    contrato = contratos[0]

    print("Contrato:", contrato.get("instId"))
    print("Estado:", contrato.get("state"))
    print("Liquidación:", contrato.get("settleCcy"))

    if contrato.get("state") != "live":
        raise Exception("El contrato no está activo.")

    # 2. CONSULTAR VELAS DE UN MINUTO

    print("\n2. DESCARGANDO VELAS DE 1 MINUTO...")

    r = sesion.get(
        BASE_URL + "/api/v5/market/candles",
        params={
            "instId": SIMBOLO,
            "bar": "1m",
            "limit": "10"
        },
        timeout=15
    )

    print("HTTP velas:", r.status_code)
    r.raise_for_status()

    datos = r.json()

    if datos.get("code") != "0":
        raise Exception(datos.get("msg", "Error de velas"))

    velas = datos.get("data", [])

    if not velas:
        raise Exception("OKX no devolvió velas.")

    columnas = [
        "timestamp", "open", "high", "low",
        "close", "volume", "vol_currency",
        "vol_quote", "confirm"
    ]

    df_okx_1m = pd.DataFrame(
        velas,
        columns=columnas
    )

    df_okx_1m["timestamp"] = pd.to_datetime(
        pd.to_numeric(df_okx_1m["timestamp"]),
        unit="ms",
        utc=True
    )

    for col in [
        "open", "high", "low", "close",
        "volume", "vol_currency", "vol_quote"
    ]:
        df_okx_1m[col] = pd.to_numeric(
            df_okx_1m[col]
        )

    df_okx_1m["confirm"] = (
        df_okx_1m["confirm"].astype(str)
    )

    df_okx_1m = (
        df_okx_1m
        .sort_values("timestamp")
        .drop_duplicates("timestamp")
        .reset_index(drop=True)
    )

    # confirm = 1 significa vela cerrada
    df_okx_1m_cerrado = df_okx_1m[
        df_okx_1m["confirm"] == "1"
    ].copy()

    print("\n3. RESULTADO")

    print("Velas recibidas:", len(df_okx_1m))
    print("Velas cerradas:", len(df_okx_1m_cerrado))

    if df_okx_1m_cerrado.empty:
        raise Exception("No hay velas cerradas.")

    ultima = df_okx_1m_cerrado.iloc[-1]

    print("\nULTIMA VELA CERRADA")
    print("Hora UTC:", ultima["timestamp"])
    print("Open:", ultima["open"])
    print("High:", ultima["high"])
    print("Low:", ultima["low"])
    print("Close:", ultima["close"])
    print("Volume:", ultima["volume"])

    antiguedad = (
        pd.Timestamp.now(tz="UTC")
        - (ultima["timestamp"] + pd.Timedelta(minutes=1))
    ).total_seconds()

    print(
        "\nAntigüedad de la vela:",
        round(antiguedad, 1),
        "segundos"
    )

    if -60 <= antiguedad <= 180:
        print("ESTADO: DATOS RECIENTES")
    else:
        print("ESTADO: REVISAR ANTIGÜEDAD")

    print("\n======================================")
    print("OKX: CONSULTA COMPLETADA")
    print("SOLO LECTURA — SIN ORDENES")
    print("======================================")

except requests.exceptions.RequestException as error:
    print("\nERROR DE CONEXION:")
    print(str(error))

except Exception as error:
    print("\nDIAGNOSTICO:")
    print(str(error))

finally:
    sesion.close()


CRYPTOBOT AI — DIAGNOSTICO OKX

1. VERIFICANDO CONTRATO...
HTTP instrumentos: 200
Contrato: BABY-USDT-SWAP
Estado: live
Liquidación: USDT

2. DESCARGANDO VELAS DE 1 MINUTO...
HTTP velas: 200

3. RESULTADO
Velas recibidas: 10
Velas cerradas: 9

ULTIMA VELA CERRADA
Hora UTC: 2026-11-25 05:39:00+00:00
Open: 0.008641
High: 0.008656
Low: 0.008639
Close: 0.008652
Volume: 3044

Antigüedad de la vela: 35.0 segundos
ESTADO: DATOS RECIENTES

OKX: CONSULTA COMPLETADA
SOLO LECTURA — SIN ORDENES


In [ ]:

# ============================================================
# CRYPTOBOT AI — BABYUSDT
# MODULO 3.5 — ACTUALIZACION AUTOMATICA OKX
# Frecuencia: 60 segundos
# SOLO LECTURA — SIN ORDENES
# ============================================================

import requests
import pandas as pd
import time
from datetime import datetime, timezone

BASE_URL = "https://www.okx.com"
SIMBOLO = "BABY-USDT-SWAP"

TEMPORALIDADES = {
    "1m": "1m",
    "5m": "5m",
    "15m": "15m",
    "1h": "1H",
    "4h": "4H"
}

FRECUENCIA = 60
HISTORIAL_INICIAL = 200

sesion_okx = requests.Session()
sesion_okx.headers.update({
    "User-Agent": "CryptoBot-AI-ReadOnly/1.0"
})

okx_data = {}

COLUMNAS = [
    "timestamp", "open", "high", "low",
    "close", "volume", "vol_currency",
    "vol_quote", "confirm"
]


def descargar_velas(intervalo, limite):

    url = BASE_URL + "/api/v5/market/candles"

    respuesta = sesion_okx.get(
        url,
        params={
            "instId": SIMBOLO,
            "bar": intervalo,
            "limit": str(limite)
        },
        timeout=20
    )

    respuesta.raise_for_status()

    resultado = respuesta.json()

    if resultado.get("code") != "0":
        raise Exception(resultado.get("msg", "Error OKX"))

    datos = resultado.get("data", [])

    if not datos:
        raise Exception("OKX no devolvió velas.")

    df = pd.DataFrame(datos, columns=COLUMNAS)

    df["timestamp"] = pd.to_datetime(
        pd.to_numeric(df["timestamp"]),
        unit="ms",
        utc=True
    )

    for col in [
        "open", "high", "low", "close",
        "volume", "vol_currency", "vol_quote"
    ]:
        df[col] = pd.to_numeric(df[col])

    df["confirm"] = df["confirm"].astype(str)

    df = (
        df.sort_values("timestamp")
        .drop_duplicates("timestamp")
        .reset_index(drop=True)
    )

    # Conservar solamente velas cerradas
    df = df[df["confirm"] == "1"].copy()

    return df


def calcular_indicadores(df):

    df = df.copy()

    df["EMA21"] = df["close"].ewm(
        span=21, adjust=False
    ).mean()

    df["EMA50"] = df["close"].ewm(
        span=50, adjust=False
    ).mean()

    cambio = df["close"].diff()

    ganancias = cambio.clip(lower=0)
    perdidas = -cambio.clip(upper=0)

    media_ganancias = ganancias.ewm(
        alpha=1/14, adjust=False
    ).mean()

    media_perdidas = perdidas.ewm(
        alpha=1/14, adjust=False
    ).mean()

    rs = media_ganancias / media_perdidas.replace(0, float("nan"))

    df["RSI14"] = 100 - (100 / (1 + rs))

    rango1 = df["high"] - df["low"]
    rango2 = (df["high"] - df["close"].shift()).abs()
    rango3 = (df["low"] - df["close"].shift()).abs()

    rango_real = pd.concat(
        [rango1, rango2, rango3],
        axis=1
    ).max(axis=1)

    df["ATR14"] = rango_real.ewm(
        alpha=1/14, adjust=False
    ).mean()

    return df


def clasificar_tendencia(ultima):

    if (
        ultima["EMA21"] > ultima["EMA50"]
        and ultima["close"] > ultima["EMA21"]
        and ultima["RSI14"] > 50
    ):
        return "ALCISTA"

    elif (
        ultima["EMA21"] < ultima["EMA50"]
        and ultima["close"] < ultima["EMA21"]
        and ultima["RSI14"] < 50
    ):
        return "BAJISTA"

    return "NEUTRAL"


def actualizar_temporalidad(nombre, intervalo, inicial=False):

    if inicial:
        nuevas = descargar_velas(
            intervalo,
            HISTORIAL_INICIAL
        )
    else:
        nuevas = descargar_velas(intervalo, 3)

    anterior = okx_data[nombre]

    combinado = pd.concat(
        [anterior, nuevas],
        ignore_index=True
    )

    combinado = (
        combinado.sort_values("timestamp")
        .drop_duplicates("timestamp", keep="last")
        .reset_index(drop=True)
    )

    okx_data[nombre] = combinado.tail(500).reset_index(drop=True)

    return len(nuevas)


def mostrar_analisis():

    print("\n========================================")
    print("CRYPTOBOT AI — ANALISIS ACTUALIZADO")
    print("========================================")

    for nombre in TEMPORALIDADES:

        df = okx_data[nombre]

        if len(df) < 50:
            print(nombre, "— HISTORIAL INSUFICIENTE")
            continue

        analisis = calcular_indicadores(df)
        ultima = analisis.iloc[-1]

        tendencia = clasificar_tendencia(ultima)

        print("\nTEMPORALIDAD:", nombre.upper())
        print("Última vela:", ultima["timestamp"])
        print("Precio:", ultima["close"])
        print("EMA21:", round(ultima["EMA21"], 8))
        print("EMA50:", round(ultima["EMA50"], 8))
        print("RSI14:", round(ultima["RSI14"], 2))
        print("ATR14:", round(ultima["ATR14"], 8))
        print("Tendencia:", tendencia)

    print("\nFuente: OKX Futures")
    print("Mercado: BABY-USDT-SWAP")
    print("Modo: SOLO LECTURA")
    print("No se enviaron órdenes.")


# ============================================================
# 1. DESCARGA INICIAL
# ============================================================

print("========================================")
print("CRYPTOBOT AI — MODULO 3.5")
print("ACTUALIZACION AUTOMATICA")
print("========================================")

print("\n1. DESCARGANDO HISTORIAL INICIAL...")

for nombre, intervalo in TEMPORALIDADES.items():

    df = descargar_velas(
        intervalo,
        HISTORIAL_INICIAL
    )

    okx_data[nombre] = df

    print(
        nombre,
        "—",
        len(df),
        "velas cerradas"
    )

print("\n2. HISTORIAL INICIAL COMPLETADO")

mostrar_analisis()


# ============================================================
# 2. CICLO AUTOMATICO
# ============================================================

print("\n========================================")
print("ACTUALIZACION AUTOMATICA ACTIVADA")
print("Frecuencia: cada 60 segundos")
print("Para detener: pulsa el botón STOP de Colab")
print("========================================")

ciclo = 0

try:

    while True:

        ciclo += 1

        print("\n----------------------------------------")
        print("CICLO:", ciclo)
        print("Hora UTC:", datetime.now(timezone.utc))
        print("----------------------------------------")

        actualizaciones = 0

        for nombre, intervalo in TEMPORALIDADES.items():

            try:

                nuevas = actualizar_temporalidad(
                    nombre,
                    intervalo
                )

                actualizaciones += 1

                print(
                    nombre,
                    "— consulta correcta;",
                    nuevas,
                    "velas recibidas"
                )

            except Exception as error:

                print(
                    nombre,
                    "— ERROR:",
                    str(error)[:180]
                )

        if actualizaciones > 0:
            mostrar_analisis()
        else:
            print("No se pudo actualizar ninguna temporalidad.")

        print("\nPróxima actualización en 60 segundos.")

        time.sleep(FRECUENCIA)

except KeyboardInterrupt:

    print("\nACTUALIZACION DETENIDA POR EL USUARIO.")

finally:

    sesion_okx.close()

    print("\n========================================")
    print("MODULO 3.5 FINALIZADO")
    print("No se enviaron órdenes.")
    print("========================================")


CRYPTOBOT AI — MODULO 3.5
ACTUALIZACION AUTOMATICA

1. DESCARGANDO HISTORIAL INICIAL...
1m — 199 velas cerradas
5m — 199 velas cerradas
15m — 199 velas cerradas
1h — 199 velas cerradas
4h — 199 velas cerradas

2. HISTORIAL INICIAL COMPLETADO

CRYPTOBOT AI — ANALISIS ACTUALIZADO

TEMPORALIDAD: 1M
Última vela: 2026-11-25 05:41:00+00:00
Precio: 0.008651
EMA21: 0.00864583
EMA50: 0.00862481
RSI14: 52.3
ATR14: 2.347e-05
Tendencia: ALCISTA

TEMPORALIDAD: 5M
Última vela: 2026-11-25 05:35:00+00:00
Precio: 0.008652
EMA21: 0.00857684
EMA50: 0.00848736
RSI14: 61.27
ATR14: 6.45e-05
Tendencia: ALCISTA

TEMPORALIDAD: 15M
Última vela: 2026-11-25 05:15:00+00:00
Precio: 0.008623
EMA21: 0.00845424
EMA50: 0.00841646
RSI14: 68.92
ATR14: 9.513e-05
Tendencia: ALCISTA

TEMPORALIDAD: 1H
Última vela: 2026-11-25 04:00:00+00:00
Precio: 0.008634
EMA21: 0.00841083
EMA50: 0.00830018
RSI14: 59.77
ATR14: 0.0002138
Tendencia: ALCISTA

TEMPORALIDAD: 4H
Última vela: 2026-11-25 00:00:00+00:00
Precio: 0.008404
EMA21: 0.008

In [ ]:

# ============================================================
# CRYPTOBOT AI — BABYUSDT
# MODULO 3.6 — GESTOR DE DATOS E HISTORIAL
# Proveedor: OKX Futures
# Actualizacion: 60 segundos
# SOLO LECTURA — SIN ORDENES
# ============================================================

import requests
import pandas as pd
import time
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONFIGURACION
# ------------------------------------------------------------

BASE_URL = "https://www.okx.com"
SIMBOLO = "BABY-USDT-SWAP"

FRECUENCIA = 60
VELAS_INICIALES = 200
MAX_HISTORIAL = 1000

TEMPORALIDADES = {
    "1m": ("1m", 60),
    "5m": ("5m", 300),
    "15m": ("15m", 900),
    "1h": ("1H", 3600),
    "4h": ("4H", 14400)
}

COLUMNAS = [
    "timestamp", "open", "high", "low",
    "close", "volume", "vol_currency",
    "vol_quote", "confirm"
]

# Historial independiente del modulo 3.5
okx_data_36 = {}

sesion_36 = requests.Session()
sesion_36.headers.update({
    "User-Agent": "CryptoBot-AI-ReadOnly/1.0"
})


# ------------------------------------------------------------
# 2. DESCARGAR VELAS CERRADAS
# ------------------------------------------------------------

def m36_descargar_velas(intervalo, limite=200):

    url = BASE_URL + "/api/v5/market/candles"

    respuesta = sesion_36.get(
        url,
        params={
            "instId": SIMBOLO,
            "bar": intervalo,
            "limit": str(limite)
        },
        timeout=20
    )

    respuesta.raise_for_status()

    resultado = respuesta.json()

    if resultado.get("code") != "0":
        raise Exception(resultado.get("msg", "Error OKX"))

    datos = resultado.get("data", [])

    if not datos:
        raise Exception("No se recibieron velas.")

    df = pd.DataFrame(datos, columns=COLUMNAS)

    df["timestamp"] = pd.to_datetime(
        pd.to_numeric(df["timestamp"]),
        unit="ms",
        utc=True
    )

    for columna in [
        "open", "high", "low", "close",
        "volume", "vol_currency", "vol_quote"
    ]:
        df[columna] = pd.to_numeric(df[columna])

    df["confirm"] = df["confirm"].astype(str)

    df = (
        df.sort_values("timestamp")
        .drop_duplicates("timestamp")
        .reset_index(drop=True)
    )

    # Conservar únicamente velas cerradas
    df = df[df["confirm"] == "1"].copy()

    return df.reset_index(drop=True)


# ------------------------------------------------------------
# 3. COMBINAR HISTORIAL SIN DUPLICADOS
# ------------------------------------------------------------

def m36_combinar_historial(nombre, nuevas):

    anterior = okx_data_36.get(nombre)

    if anterior is None or anterior.empty:
        combinado = nuevas.copy()
    else:
        combinado = pd.concat(
            [anterior, nuevas],
            ignore_index=True
        )

    combinado = (
        combinado
        .sort_values("timestamp")
        .drop_duplicates("timestamp", keep="last")
        .reset_index(drop=True)
    )

    # Mantener como máximo 1000 velas
    combinado = combinado.tail(
        MAX_HISTORIAL
    ).reset_index(drop=True)

    okx_data_36[nombre] = combinado

    return combinado


# ------------------------------------------------------------
# 4. CONTROLAR CONTINUIDAD Y ACTUALIZACION
# ------------------------------------------------------------

def m36_estado_datos(nombre, intervalo_segundos):

    df = okx_data_36[nombre]

    if len(df) < 2:
        return 0, "HISTORIAL INSUFICIENTE", None

    diferencias = (
        df["timestamp"]
        .diff()
        .dropna()
        .dt.total_seconds()
    )

    huecos = int(
        (diferencias > intervalo_segundos * 1.5).sum()
    )

    ultima_apertura = df["timestamp"].iloc[-1]

    hora_cierre = (
        ultima_apertura
        + pd.Timedelta(seconds=intervalo_segundos)
    )

    ahora = pd.Timestamp.now(tz="UTC")

    antiguedad = (
        ahora - hora_cierre
    ).total_seconds()

    limite = intervalo_segundos + 180

    if antiguedad < -60:
        estado = "RELOJ / DATOS POR VERIFICAR"
    elif antiguedad > limite:
        estado = "DATOS ANTIGUOS"
    elif huecos > 0:
        estado = "REVISAR HUECOS"
    else:
        estado = "DATOS ACTUALIZADOS"

    return huecos, estado, antiguedad


# ------------------------------------------------------------
# 5. MOSTRAR RESUMEN
# ------------------------------------------------------------

def m36_mostrar_resumen():

    print("\n========================================")
    print("CRYPTOBOT AI — ESTADO DEL HISTORIAL")
    print("========================================")

    for nombre, (intervalo, segundos) in TEMPORALIDADES.items():

        df = okx_data_36[nombre]

        huecos, estado, antiguedad = m36_estado_datos(
            nombre,
            segundos
        )

        ultima = df.iloc[-1]

        print("\nTEMPORALIDAD:", nombre.upper())
        print("Velas almacenadas:", len(df))
        print("Última vela abierta:", ultima["timestamp"])
        print("Último cierre:", ultima["close"])
        print("Huecos detectados:", huecos)

        if antiguedad is not None:
            print(
                "Antigüedad desde cierre:",
                round(antiguedad, 1),
                "segundos"
            )

        print("Estado:", estado)


# ------------------------------------------------------------
# 6. DESCARGA INICIAL
# ------------------------------------------------------------

print("========================================")
print("CRYPTOBOT AI — MODULO 3.6")
print("GESTOR DE DATOS E HISTORIAL")
print("========================================")

print("\n1. CARGANDO HISTORIAL INICIAL...")

for nombre, (intervalo, segundos) in TEMPORALIDADES.items():

    try:

        datos = m36_descargar_velas(
            intervalo,
            VELAS_INICIALES
        )

        okx_data_36[nombre] = datos

        print(
            nombre,
            "—",
            len(datos),
            "velas cerradas"
        )

    except Exception as error:

        print(nombre, "— ERROR:", str(error)[:180])

if not okx_data_36:
    sesion_36.close()
    raise Exception("No se pudo descargar ningún historial.")

m36_mostrar_resumen()


# ------------------------------------------------------------
# 7. ACTUALIZACION AUTOMATICA
# ------------------------------------------------------------

print("\n========================================")
print("GESTOR AUTOMATICO ACTIVADO")
print("Frecuencia: 60 segundos")
print("Para detener: botón STOP de Colab")
print("========================================")

ciclo_36 = 0

try:

    while True:

        ciclo_36 += 1

        print("\n----------------------------------------")
        print("ACTUALIZACION:", ciclo_36)
        print("Hora UTC:", datetime.now(timezone.utc))
        print("----------------------------------------")

        exitos = 0

        for nombre, (intervalo, segundos) in TEMPORALIDADES.items():

            try:

                nuevas = m36_descargar_velas(
                    intervalo,
                    200
                )

                anterior = okx_data_36.get(nombre)

                ultima_anterior = (
                    anterior["timestamp"].iloc[-1]
                    if anterior is not None and not anterior.empty
                    else None
                )

                combinado = m36_combinar_historial(
                    nombre,
                    nuevas
                )

                ultima_nueva = combinado["timestamp"].iloc[-1]

                if ultima_anterior is None:
                    cambio = "Historial inicial"
                elif ultima_nueva > ultima_anterior:
                    cambio = "VELA NUEVA"
                else:
                    cambio = "Sin vela nueva cerrada"

                exitos += 1

                print(
                    nombre,
                    "—",
                    cambio,
                    "| Total:",
                    len(combinado)
                )

            except Exception as error:

                print(
                    nombre,
                    "— ERROR:",
                    str(error)[:180]
                )

        if exitos > 0:
            m36_mostrar_resumen()

        print("\nPróxima actualización en 60 segundos.")

        time.sleep(FRECUENCIA)

except KeyboardInterrupt:

    print("\nACTUALIZACION DETENIDA.")

finally:

    sesion_36.close()

    print("\n========================================")
    print("MODULO 3.6 FINALIZADO")
    print("Historial conservado en memoria.")
    print("No se enviaron órdenes.")
    print("========================================")


CRYPTOBOT AI — MODULO 3.6
GESTOR DE DATOS E HISTORIAL

1. CARGANDO HISTORIAL INICIAL...
1m — 199 velas cerradas
5m — 199 velas cerradas
15m — 199 velas cerradas
1h — 199 velas cerradas
4h — 199 velas cerradas

CRYPTOBOT AI — ESTADO DEL HISTORIAL

TEMPORALIDAD: 1M
Velas almacenadas: 199
Última vela abierta: 2026-11-25 05:51:00+00:00
Último cierre: 0.008812
Huecos detectados: 0
Antigüedad desde cierre: 13.3 segundos
Estado: DATOS ACTUALIZADOS

TEMPORALIDAD: 5M
Velas almacenadas: 199
Última vela abierta: 2026-11-25 05:45:00+00:00
Último cierre: 0.008765
Huecos detectados: 0
Antigüedad desde cierre: 133.3 segundos
Estado: DATOS ACTUALIZADOS

TEMPORALIDAD: 15M
Velas almacenadas: 199
Última vela abierta: 2026-11-25 05:30:00+00:00
Último cierre: 0.008679
Huecos detectados: 0
Antigüedad desde cierre: 433.3 segundos
Estado: DATOS ACTUALIZADOS

TEMPORALIDAD: 1H
Velas almacenadas: 199
Última vela abierta: 2026-11-25 04:00:00+00:00
Último cierre: 0.008634
Huecos detectados: 0
Antigüedad desde cier

In [ ]:

# ============================================================
# CRYPTOBOT AI — MODULO 3.7
# PERSISTENCIA Y RECUPERACION DEL HISTORIAL
# PAR: BABYUSDT | FUENTE: OKX | SOLO DATOS PUBLICOS
# ============================================================

import os
import time
import requests
import pandas as pd
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. CONFIGURACION
# ------------------------------------------------------------

M37_INST_ID = "BABY-USDT-SWAP"
M37_URL = "https://www.okx.com/api/v5/market/candles"

M37_TEMPORALIDADES = {
    "1m": 60,
    "5m": 300,
    "15m": 900,
    "1H": 3600,
    "4H": 14400
}

M37_INTERVALOS = {
    "1m": "1m",
    "5m": "5m",
    "15m": "15m",
    "1H": "1H",
    "4H": "4H"
}

M37_CARPETA = "/content/cryptobot_ai/BABYUSDT"
M37_MAX_HISTORIAL = 1000
M37_FRECUENCIA = 60

os.makedirs(M37_CARPETA, exist_ok=True)

m37_sesion = requests.Session()
m37_sesion.headers.update({"User-Agent": "CryptoBot-AI/3.7"})

m37_datos = {}
m37_errores = []

M37_COLUMNAS = [
    "timestamp", "open", "high", "low", "close",
    "volume", "volume_currency", "volume_quote", "confirm"
]


# ------------------------------------------------------------
# 2. DESCARGAR VELAS DESDE OKX
# ------------------------------------------------------------

def m37_descargar_velas(intervalo, limite=3):

    parametros = {
        "instId": M37_INST_ID,
        "bar": intervalo,
        "limit": str(limite)
    }

    respuesta = m37_sesion.get(
        M37_URL,
        params=parametros,
        timeout=20
    )

    respuesta.raise_for_status()
    resultado = respuesta.json()

    if resultado.get("code") != "0":
        raise Exception(
            f"OKX respondió: {resultado.get('msg', 'Error desconocido')}"
        )

    filas = resultado.get("data", [])

    if not filas:
        raise Exception("OKX no devolvió velas.")

    df = pd.DataFrame(filas, columns=M37_COLUMNAS)

    df["timestamp"] = pd.to_datetime(
        pd.to_numeric(df["timestamp"]),
        unit="ms",
        utc=True
    )

    columnas_numericas = [
        "open", "high", "low", "close",
        "volume", "volume_currency", "volume_quote"
    ]

    for columna in columnas_numericas:
        df[columna] = pd.to_numeric(
            df[columna],
            errors="coerce"
        )

    df["confirm"] = df["confirm"].astype(str)

    # Solo conservar velas cerradas
    df = df[df["confirm"] == "1"].copy()

    df = df.dropna(subset=["timestamp", "close"])

    return df.sort_values("timestamp").reset_index(drop=True)


# ------------------------------------------------------------
# 3. CARGAR HISTORIAL GUARDADO
# ------------------------------------------------------------

def m37_cargar_csv(nombre):

    ruta = os.path.join(
        M37_CARPETA,
        f"BABYUSDT_{nombre}.csv"
    )

    if not os.path.exists(ruta):
        return pd.DataFrame(columns=M37_COLUMNAS)

    try:
        df = pd.read_csv(ruta)

        df["timestamp"] = pd.to_datetime(
            df["timestamp"],
            utc=True,
            errors="coerce"
        )

        for columna in [
            "open", "high", "low", "close",
            "volume", "volume_currency", "volume_quote"
        ]:
            df[columna] = pd.to_numeric(
                df[columna],
                errors="coerce"
            )

        df["confirm"] = df["confirm"].astype(str)

        df = df.dropna(subset=["timestamp", "close"])
        df = df.sort_values("timestamp")
        df = df.drop_duplicates("timestamp", keep="last")

        return df.tail(M37_MAX_HISTORIAL).reset_index(drop=True)

    except Exception as error:
        print(f"⚠️ No se pudo leer {nombre}: {error}")
        return pd.DataFrame(columns=M37_COLUMNAS)


# ------------------------------------------------------------
# 4. COMBINAR Y GUARDAR HISTORIAL
# ------------------------------------------------------------

def m37_combinar(nombre, antiguo, nuevo):

    if antiguo is None or antiguo.empty:
        combinado = nuevo.copy()
    elif nuevo is None or nuevo.empty:
        combinado = antiguo.copy()
    else:
        combinado = pd.concat(
            [antiguo, nuevo],
            ignore_index=True
        )

    combinado = combinado.dropna(subset=["timestamp", "close"])
    combinado = combinado.sort_values("timestamp")
    combinado = combinado.drop_duplicates(
        subset=["timestamp"],
        keep="last"
    )

    return combinado.tail(
        M37_MAX_HISTORIAL
    ).reset_index(drop=True)


def m37_guardar_csv(nombre, df):

    ruta = os.path.join(
        M37_CARPETA,
        f"BABYUSDT_{nombre}.csv"
    )

    temporal = ruta + ".tmp"

    df.to_csv(temporal, index=False)
    os.replace(temporal, ruta)

    return ruta


# ------------------------------------------------------------
# 5. DETECTAR VELAS FALTANTES
# ------------------------------------------------------------

def m37_detectar_huecos(df, segundos):

    if df is None or len(df) < 2:
        return 0

    tiempos = (
        df["timestamp"]
        .sort_values()
        .drop_duplicates()
    )

    diferencias = tiempos.diff().dropna()

    faltantes = 0

    for diferencia in diferencias:
        segundos_reales = diferencia.total_seconds()

        if segundos_reales > segundos * 1.5:
            cantidad = round(
                segundos_reales / segundos
            ) - 1

            faltantes += max(0, cantidad)

    return faltantes


# ------------------------------------------------------------
# 6. INICIALIZAR O RECUPERAR HISTORIAL
# ------------------------------------------------------------

print("=" * 55)
print("CRYPTOBOT AI — MODULO 3.7")
print("PERSISTENCIA Y RECUPERACION DEL HISTORIAL")
print("=" * 55)

for nombre, segundos in M37_TEMPORALIDADES.items():

    print(f"\n📊 Inicializando temporalidad {nombre}...")

    try:
        guardado = m37_cargar_csv(nombre)

        if guardado.empty:
            print("   No existe historial guardado.")
            print("   Descargando historial inicial...")

            nuevas = m37_descargar_velas(
                M37_INTERVALOS[nombre],
                limite=200
            )

        else:
            print(f"   Historial recuperado: {len(guardado)} velas.")

            nuevas = m37_descargar_velas(
                M37_INTERVALOS[nombre],
                limite=3
            )

        combinado = m37_combinar(
            nombre,
            guardado,
            nuevas
        )

        # Si detecta huecos, intenta recuperar los más recientes
        huecos = m37_detectar_huecos(
            combinado,
            segundos
        )

        if huecos > 0:
            print(
                f"   ⚠️ Detectados aproximadamente {huecos} "
                "periodos faltantes."
            )

            print("   Intentando recuperar con hasta 300 velas...")

            recuperacion = m37_descargar_velas(
                M37_INTERVALOS[nombre],
                limite=300
            )

            combinado = m37_combinar(
                nombre,
                combinado,
                recuperacion
            )

        huecos_finales = m37_detectar_huecos(
            combinado,
            segundos
        )

        m37_datos[nombre] = combinado

        ruta = m37_guardar_csv(
            nombre,
            combinado
        )

        print(f"   Velas guardadas: {len(combinado)}")
        print(f"   Huecos pendientes: {huecos_finales}")
        print(f"   Archivo: {ruta}")

    except Exception as error:
        m37_errores.append((nombre, str(error)))
        print(f"   ❌ Error en {nombre}: {error}")


# ------------------------------------------------------------
# 7. MOSTRAR ESTADO DEL HISTORIAL
# ------------------------------------------------------------

def m37_mostrar_estado():

    print("\n" + "=" * 65)
    print("ESTADO DEL HISTORIAL — BABYUSDT")
    print("=" * 65)

    for nombre, segundos in M37_TEMPORALIDADES.items():

        df = m37_datos.get(nombre)

        if df is None or df.empty:
            print(f"{nombre}: SIN DATOS")
            continue

        ultima = df.iloc[-1]

        huecos = m37_detectar_huecos(
            df,
            segundos
        )

        print(f"\nTemporalidad: {nombre}")
        print(f"Velas almacenadas: {len(df)}")
        print(f"Último cierre: {ultima['close']}")
        print(f"Última vela cerrada: {ultima['timestamp']}")
        print(f"Huecos pendientes: {huecos}")

    print("\nCarpeta de almacenamiento:")
    print(M37_CARPETA)


m37_mostrar_estado()


# ------------------------------------------------------------
# 8. ACTUALIZACION AUTOMATICA CADA 60 SEGUNDOS
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("ACTUALIZACION AUTOMATICA ACTIVADA")
print("Presiona STOP en Colab para detener.")
print("=" * 55)

m37_ciclo = 0

try:

    while True:

        m37_ciclo += 1

        print(
            f"\n🔄 CICLO {m37_ciclo} — "
            f"{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}"
        )

        for nombre, segundos in M37_TEMPORALIDADES.items():

            try:
                nuevas = m37_descargar_velas(
                    M37_INTERVALOS[nombre],
                    limite=3
                )

                anterior = m37_datos.get(
                    nombre,
                    pd.DataFrame(columns=M37_COLUMNAS)
                )

                combinado = m37_combinar(
                    nombre,
                    anterior,
                    nuevas
                )

                huecos = m37_detectar_huecos(
                    combinado,
                    segundos
                )

                # Intento de recuperación cuando hay huecos
                if huecos > 0:
                    recuperacion = m37_descargar_velas(
                        M37_INTERVALOS[nombre],
                        limite=300
                    )

                    combinado = m37_combinar(
                        nombre,
                        combinado,
                        recuperacion
                    )

                m37_datos[nombre] = combinado

                ruta = m37_guardar_csv(
                    nombre,
                    combinado
                )

                ultima = combinado.iloc[-1]

                print(
                    f"✅ {nombre} | "
                    f"Cierre: {ultima['close']} | "
                    f"Velas: {len(combinado)} | "
                    f"Huecos: {m37_detectar_huecos(combinado, segundos)}"
                )

            except Exception as error:
                print(f"❌ {nombre}: {error}")

        print("💾 Historial actualizado y guardado.")

        time.sleep(M37_FRECUENCIA)

except KeyboardInterrupt:
    print("\n🛑 Módulo detenido manualmente.")

finally:
    m37_sesion.close()
    print("Conexión cerrada.")
    print("Los archivos CSV guardados permanecen en la carpeta de sesión.")


CRYPTOBOT AI — MODULO 3.7
PERSISTENCIA Y RECUPERACION DEL HISTORIAL

📊 Inicializando temporalidad 1m...
   Historial recuperado: 200 velas.
   Velas guardadas: 200
   Huecos pendientes: 0
   Archivo: /content/cryptobot_ai/BABYUSDT/BABYUSDT_1m.csv

📊 Inicializando temporalidad 5m...
   Historial recuperado: 199 velas.
   Velas guardadas: 199
   Huecos pendientes: 0
   Archivo: /content/cryptobot_ai/BABYUSDT/BABYUSDT_5m.csv

📊 Inicializando temporalidad 15m...
   Historial recuperado: 199 velas.
   Velas guardadas: 199
   Huecos pendientes: 0
   Archivo: /content/cryptobot_ai/BABYUSDT/BABYUSDT_15m.csv

📊 Inicializando temporalidad 1H...
   Historial recuperado: 199 velas.
   Velas guardadas: 199
   Huecos pendientes: 0
   Archivo: /content/cryptobot_ai/BABYUSDT/BABYUSDT_1H.csv

📊 Inicializando temporalidad 4H...
   Historial recuperado: 199 velas.
   Velas guardadas: 199
   Huecos pendientes: 0
   Archivo: /content/cryptobot_ai/BABYUSDT/BABYUSDT_4H.csv

ESTADO DEL HISTORIAL — BABYUSDT

T

In [ ]:

# ============================================================
# CRYPTOBOT AI — MODULO 3.8
# ALMACENAMIENTO PERMANENTE EN GOOGLE DRIVE
# BABYUSDT | SOLO DATOS | SIN ORDENES
# ============================================================

import os
import shutil
import pandas as pd
from datetime import datetime, timezone
from google.colab import drive

print("=" * 60)
print("CRYPTOBOT AI — MODULO 3.8")
print("ALMACENAMIENTO PERMANENTE EN GOOGLE DRIVE")
print("=" * 60)

# ------------------------------------------------------------
# 1. CONECTAR GOOGLE DRIVE
# ------------------------------------------------------------

print("\nConectando Google Drive...")

drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. CONFIGURAR CARPETAS
# ------------------------------------------------------------

M38_CARPETA_LOCAL = "/content/cryptobot_ai/BABYUSDT"

M38_CARPETA_DRIVE = (
    "/content/drive/MyDrive/"
    "CryptoBot_AI/BABYUSDT"
)

os.makedirs(M38_CARPETA_LOCAL, exist_ok=True)
os.makedirs(M38_CARPETA_DRIVE, exist_ok=True)

M38_TEMPORALIDADES = ["1m", "5m", "15m", "1H", "4H"]

M38_COLUMNAS = [
    "timestamp", "open", "high", "low", "close",
    "volume", "volume_currency", "volume_quote", "confirm"
]

# ------------------------------------------------------------
# 3. LEER Y NORMALIZAR ARCHIVOS
# ------------------------------------------------------------

def m38_leer_csv(ruta):

    if not os.path.exists(ruta):
        return pd.DataFrame(columns=M38_COLUMNAS)

    df = pd.read_csv(ruta)

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        utc=True,
        errors="coerce"
    )

    for columna in M38_COLUMNAS:
        if columna not in df.columns:
            df[columna] = None

    for columna in [
        "open", "high", "low", "close",
        "volume", "volume_currency", "volume_quote"
    ]:
        df[columna] = pd.to_numeric(
            df[columna],
            errors="coerce"
        )

    df["confirm"] = df["confirm"].astype(str)

    df = df.dropna(
        subset=["timestamp", "close"]
    )

    df = df[M38_COLUMNAS]

    df = df.sort_values("timestamp")
    df = df.drop_duplicates(
        subset=["timestamp"],
        keep="last"
    )

    return df.reset_index(drop=True)


# ------------------------------------------------------------
# 4. COMBINAR HISTORIAL LOCAL Y DRIVE
# ------------------------------------------------------------

def m38_combinar(local, drive_df):

    if local.empty and drive_df.empty:
        return pd.DataFrame(columns=M38_COLUMNAS)

    combinado = pd.concat(
        [local, drive_df],
        ignore_index=True
    )

    combinado = combinado.dropna(
        subset=["timestamp", "close"]
    )

    combinado = combinado.sort_values("timestamp")

    combinado = combinado.drop_duplicates(
        subset=["timestamp"],
        keep="last"
    )

    # Mantener el límite del proyecto
    combinado = combinado.tail(1000)

    return combinado.reset_index(drop=True)


# ------------------------------------------------------------
# 5. GUARDAR COPIA PERMANENTE
# ------------------------------------------------------------

def m38_guardar_archivo(nombre, df):

    nombre_archivo = f"BABYUSDT_{nombre}.csv"

    ruta_local = os.path.join(
        M38_CARPETA_LOCAL,
        nombre_archivo
    )

    ruta_drive = os.path.join(
        M38_CARPETA_DRIVE,
        nombre_archivo
    )

    temporal = ruta_drive + ".tmp"

    df.to_csv(temporal, index=False)

    os.replace(temporal, ruta_drive)

    # Mantener también actualizada la copia local
    df.to_csv(ruta_local, index=False)

    return ruta_drive


# ------------------------------------------------------------
# 6. RECUPERAR Y SINCRONIZAR LAS CINCO TEMPORALIDADES
# ------------------------------------------------------------

print("\nSincronizando historiales...")

M38_RESUMEN = []

for nombre in M38_TEMPORALIDADES:

    archivo = f"BABYUSDT_{nombre}.csv"

    ruta_local = os.path.join(
        M38_CARPETA_LOCAL,
        archivo
    )

    ruta_drive = os.path.join(
        M38_CARPETA_DRIVE,
        archivo
    )

    try:

        datos_locales = m38_leer_csv(ruta_local)
        datos_drive = m38_leer_csv(ruta_drive)

        combinado = m38_combinar(
            datos_locales,
            datos_drive
        )

        if combinado.empty:
            print(f"⚠️ {nombre}: no hay historial disponible.")
            continue

        ruta_final = m38_guardar_archivo(
            nombre,
            combinado
        )

        ultima = combinado.iloc[-1]

        M38_RESUMEN.append({
            "Temporalidad": nombre,
            "Velas": len(combinado),
            "Último cierre": ultima["close"],
            "Última vela": str(ultima["timestamp"]),
            "Estado": "GUARDADO"
        })

        print(
            f"✅ {nombre}: {len(combinado)} velas "
            f"| Último cierre: {ultima['close']}"
        )

    except Exception as error:

        print(f"❌ Error en {nombre}: {error}")


# ------------------------------------------------------------
# 7. MOSTRAR RESULTADO
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RESULTADO DE ALMACENAMIENTO")
print("=" * 60)

if M38_RESUMEN:

    resumen = pd.DataFrame(M38_RESUMEN)

    display(resumen)

else:

    print("No se pudo guardar ningún historial.")

print("\nCarpeta permanente:")
print(M38_CARPETA_DRIVE)

print("\nCarpeta local:")
print(M38_CARPETA_LOCAL)

print("\nMódulo 3.8 finalizado.")
print("No se han enviado órdenes de trading.")


CRYPTOBOT AI — MODULO 3.8
ALMACENAMIENTO PERMANENTE EN GOOGLE DRIVE

Conectando Google Drive...


ValueError: mount failed

In [ ]:

# =====================================================
# CRYPTOBOT AI - MODULO 4
# INDICADORES TECNICOS BABYUSDT
# EMA 21 / EMA 50 / RSI 14 / ATR 14
# =====================================================

import pandas as pd
import numpy as np

def calcular_indicadores(df):

    df = df.copy()

    # EMA 21 Y EMA 50
    df["EMA_21"] = df["close"].ewm(
        span=21,
        adjust=False
    ).mean()

    df["EMA_50"] = df["close"].ewm(
        span=50,
        adjust=False
    ).mean()

    # RSI 14
    cambio = df["close"].diff()

    ganancias = cambio.clip(lower=0)
    perdidas = -cambio.clip(upper=0)

    promedio_ganancias = ganancias.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()

    promedio_perdidas = perdidas.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()

    rs = promedio_ganancias / promedio_perdidas

    df["RSI_14"] = 100 - (100 / (1 + rs))

    # ATR 14
    cierre_anterior = df["close"].shift(1)

    rango_real = pd.concat(
        [
            df["high"] - df["low"],
            (df["high"] - cierre_anterior).abs(),
            (df["low"] - cierre_anterior).abs()
        ],
        axis=1
    ).max(axis=1)

    df["ATR_14"] = rango_real.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()

    return df


# CALCULAR EN LAS CUATRO TEMPORALIDADES

baby_4h = calcular_indicadores(baby_4h)
baby_1h = calcular_indicadores(baby_1h)
baby_15m = calcular_indicadores(baby_15m)
baby_5m = calcular_indicadores(baby_5m)


# INFORME DE RESULTADOS

print("========================================")
print("CRYPTOBOT AI - MODULO 4 COMPLETADO")
print("========================================")

for nombre, df in {
    "4H": baby_4h,
    "1H": baby_1h,
    "15M": baby_15m,
    "5M": baby_5m
}.items():

    ultima = df.iloc[-1]

    print("\nTEMPORALIDAD:", nombre)
    print("Precio:", ultima["close"])
    print("EMA 21:", round(ultima["EMA_21"], 8))
    print("EMA 50:", round(ultima["EMA_50"], 8))
    print("RSI 14:", round(ultima["RSI_14"], 2))
    print("ATR 14:", round(ultima["ATR_14"], 8))

print("\nMODULO 4 LISTO")


CRYPTOBOT AI - MODULO 4 COMPLETADO

TEMPORALIDAD: 4H
Precio: 0.008481
EMA 21: 0.00792269
EMA 50: 0.00821388
RSI 14: 65.95
ATR 14: 0.00031157

TEMPORALIDAD: 1H
Precio: 0.008481
EMA 21: 0.0080999
EMA 50: 0.00791568
RSI 14: 65.76
ATR 14: 0.00021386

TEMPORALIDAD: 15M
Precio: 0.008481
EMA 21: 0.00834908
EMA 50: 0.00818132
RSI 14: 60.06
ATR 14: 0.00013016

TEMPORALIDAD: 5M
Precio: 0.008481
EMA 21: 0.00841446
EMA 50: 0.00836541
RSI 14: 61.17
ATR 14: 5.519e-05

MODULO 4 LISTO


In [ ]:

# =====================================================
# CRYPTOBOT AI - MODULO 5
# MOTOR DE SEÑALES LONG / SHORT
# PAR: BABYUSDT
# TEMPORALIDADES: 4H / 1H / 15M / 5M
# SOLO ANALISIS - SIN ORDENES REALES
# =====================================================

import pandas as pd

# 1. CLASIFICAR TENDENCIA

def analizar_tendencia(df):

    vela = df.iloc[-1]

    precio = vela["close"]
    ema21 = vela["EMA_21"]
    ema50 = vela["EMA_50"]
    rsi = vela["RSI_14"]

    if pd.isna(rsi) or pd.isna(ema21) or pd.isna(ema50):
        return "SIN DATOS"

    if (
        ema21 > ema50
        and precio > ema21
        and rsi > 50
    ):
        return "ALCISTA"

    elif (
        ema21 < ema50
        and precio < ema21
        and rsi < 50
    ):
        return "BAJISTA"

    else:
        return "NEUTRAL"


# 2. ANALIZAR LAS CUATRO TEMPORALIDADES

tendencias = {
    "4H": analizar_tendencia(baby_4h),
    "1H": analizar_tendencia(baby_1h),
    "15M": analizar_tendencia(baby_15m),
    "5M": analizar_tendencia(baby_5m)
}


# 3. MOTOR DE DECISION

def generar_senal(tendencias):

    cuatro_h = tendencias["4H"]
    una_h = tendencias["1H"]
    quince_m = tendencias["15M"]
    cinco_m = tendencias["5M"]

    # LONG: todas las temporalidades alcistas
    if all(
        t == "ALCISTA"
        for t in [cuatro_h, una_h, quince_m, cinco_m]
    ):
        return (
            "LONG",
            "Las cuatro temporalidades confirman tendencia alcista."
        )

    # SHORT: todas las temporalidades bajistas
    elif all(
        t == "BAJISTA"
        for t in [cuatro_h, una_h, quince_m, cinco_m]
    ):
        return (
            "SHORT",
            "Las cuatro temporalidades confirman tendencia bajista."
        )

    # Si hay contradicciones, esperar
    else:

        if cuatro_h == "NEUTRAL":
            motivo = (
                "4H no confirma una tendencia definida. "
                "Esperar confirmacion de la tendencia principal."
            )

        elif una_h != cuatro_h:
            motivo = (
                "La temporalidad 1H no confirma la direccion de 4H."
            )

        elif quince_m != cuatro_h:
            motivo = (
                "15M no confirma la direccion de la tendencia principal."
            )

        elif cinco_m != cuatro_h:
            motivo = (
                "5M no confirma la direccion de entrada."
            )

        else:
            motivo = "Las temporalidades no presentan alineacion completa."

        return ("ESPERAR", motivo)


# 4. GENERAR RESULTADO

senal, motivo = generar_senal(tendencias)


# 5. MOSTRAR INFORME

print("============================================")
print("CRYPTOBOT AI - MODULO 5")
print("MOTOR DE SEÑALES")
print("============================================")

print("\nPAR: BABYUSDT")

print("\nTENDENCIAS DETECTADAS:")

for temporalidad, tendencia in tendencias.items():

    print(
        temporalidad,
        "->",
        tendencia
    )

print("\n============================================")
print("DECISION DEL BOT")
print("============================================")

print("SENAL:", senal)
print("MOTIVO:", motivo)

print("\nPRECIO DE REFERENCIA:")

print(
    "BABYUSDT:",
    baby_5m["close"].iloc[-1],
    "USDT"
)

print("\nFECHA DE LA ULTIMA VELA:")

print(
    baby_5m["open_time"].iloc[-1]
)

print("\n============================================")
print("MODULO 5 COMPLETADO")
print("MODO: ANALISIS HISTORICO")
print("ORDENES REALES: DESACTIVADAS")
print("============================================")


CRYPTOBOT AI - MODULO 5
MOTOR DE SEÑALES

PAR: BABYUSDT

TENDENCIAS DETECTADAS:
4H -> NEUTRAL
1H -> ALCISTA
15M -> ALCISTA
5M -> ALCISTA

DECISION DEL BOT
SENAL: ESPERAR
MOTIVO: 4H no confirma una tendencia definida. Esperar confirmacion de la tendencia principal.

PRECIO DE REFERENCIA:
BABYUSDT: 0.008481 USDT

FECHA DE LA ULTIMA VELA:
2026-11-23 23:55:00+00:00

MODULO 5 COMPLETADO
MODO: ANALISIS HISTORICO
ORDENES REALES: DESACTIVADAS


In [ ]:

# =====================================================
# CRYPTOBOT AI - MODULO 6
# ESTRUCTURA DE MERCADO SMC
# BOS / LIQUIDEZ / FVG
# BABYUSDT
# =====================================================

import pandas as pd

# ANALIZADOR DE ESTRUCTURA
def analizar_estructura(df, ventana=5):

    df = df.copy().reset_index(drop=True)

    # Maximos y minimos anteriores, sin incluir vela actual
    df["maximo_anterior"] = (
        df["high"].shift(1).rolling(ventana).max()
    )

    df["minimo_anterior"] = (
        df["low"].shift(1).rolling(ventana).min()
    )

    # BOS: ruptura confirmada por cierre
    df["BOS_ALCISTA"] = (
        df["close"] > df["maximo_anterior"]
    )

    df["BOS_BAJISTA"] = (
        df["close"] < df["minimo_anterior"]
    )

    # BARRIDO DE LIQUIDEZ
    df["BARRIDO_MAXIMOS"] = (
        (df["high"] > df["maximo_anterior"]) &
        (df["close"] < df["maximo_anterior"])
    )

    df["BARRIDO_MINIMOS"] = (
        (df["low"] < df["minimo_anterior"]) &
        (df["close"] > df["minimo_anterior"])
    )

    # FVG ALCISTA Y BAJISTA
    df["FVG_ALCISTA"] = (
        df["low"] > df["high"].shift(2)
    )

    df["FVG_BAJISTA"] = (
        df["high"] < df["low"].shift(2)
    )

    return df


# ANALIZAR TEMPORALIDADES
baby_1h_smc = analizar_estructura(baby_1h)
baby_15m_smc = analizar_estructura(baby_15m)
baby_5m_smc = analizar_estructura(baby_5m)


# INFORME DE LA ULTIMA VELA
def mostrar_estructura(nombre, df):

    ultima = df.iloc[-1]

    print("\n================================")
    print("ESTRUCTURA:", nombre)
    print("================================")

    print(
        "BOS alcista:",
        bool(ultima["BOS_ALCISTA"])
    )

    print(
        "BOS bajista:",
        bool(ultima["BOS_BAJISTA"])
    )

    print(
        "Barrido de maximos:",
        bool(ultima["BARRIDO_MAXIMOS"])
    )

    print(
        "Barrido de minimos:",
        bool(ultima["BARRIDO_MINIMOS"])
    )

    print(
        "FVG alcista:",
        bool(ultima["FVG_ALCISTA"])
    )

    print(
        "FVG bajista:",
        bool(ultima["FVG_BAJISTA"])
    )


mostrar_estructura("1H", baby_1h_smc)
mostrar_estructura("15M", baby_15m_smc)
mostrar_estructura("5M", baby_5m_smc)


print("\n================================")
print("CRYPTOBOT AI - MODULO 6 LISTO")
print("================================")
print("Modo: Analisis historico")
print("Ordenes reales: DESACTIVADAS")



ESTRUCTURA: 1H
BOS alcista: False
BOS bajista: False
Barrido de maximos: False
Barrido de minimos: False
FVG alcista: False
FVG bajista: False

ESTRUCTURA: 15M
BOS alcista: False
BOS bajista: False
Barrido de maximos: False
Barrido de minimos: False
FVG alcista: False
FVG bajista: False

ESTRUCTURA: 5M
BOS alcista: False
BOS bajista: False
Barrido de maximos: False
Barrido de minimos: False
FVG alcista: False
FVG bajista: False

CRYPTOBOT AI - MODULO 6 LISTO
Modo: Analisis historico
Ordenes reales: DESACTIVADAS


In [ ]:

# =====================================================
# CRYPTOBOT AI - MODULO 6.1
# HISTORIAL DE EVENTOS SMC
# BOS / LIQUIDEZ / FVG
# =====================================================

import pandas as pd

def resumir_eventos(nombre, df):

    eventos = {
        "BOS_ALCISTA": "BOS alcista",
        "BOS_BAJISTA": "BOS bajista",
        "BARRIDO_MAXIMOS": "Barrido de máximos",
        "BARRIDO_MINIMOS": "Barrido de mínimos",
        "FVG_ALCISTA": "FVG alcista",
        "FVG_BAJISTA": "FVG bajista"
    }

    print("\n====================================")
    print("HISTORIAL SMC:", nombre)
    print("====================================")

    for columna, descripcion in eventos.items():

        encontrados = df[df[columna] == True]

        print("\n", descripcion)
        print("Total de eventos:", len(encontrados))

        if len(encontrados) > 0:

            recientes = encontrados.tail(5)

            for _, vela in recientes.iterrows():

                print(
                    "Fecha:",
                    vela["open_time"],
                    "| Cierre:",
                    vela["close"]
                )

        else:
            print("No se encontraron eventos.")


# REVISAR TODO EL HISTORIAL
resumir_eventos("1H", baby_1h_smc)
resumir_eventos("15M", baby_15m_smc)
resumir_eventos("5M", baby_5m_smc)


print("\n====================================")
print("MODULO 6.1 COMPLETADO")
print("====================================")



HISTORIAL SMC: 1H

 BOS alcista
Total de eventos: 50
Fecha: 2026-11-22 09:00:00+00:00 | Cierre: 0.007873
Fecha: 2026-11-23 13:00:00+00:00 | Cierre: 0.00792
Fecha: 2026-11-23 14:00:00+00:00 | Cierre: 0.007999
Fecha: 2026-11-23 17:00:00+00:00 | Cierre: 0.008285
Fecha: 2026-11-23 18:00:00+00:00 | Cierre: 0.00861

 BOS bajista
Total de eventos: 75
Fecha: 2026-11-21 21:00:00+00:00 | Cierre: 0.007455
Fecha: 2026-11-21 22:00:00+00:00 | Cierre: 0.00743
Fecha: 2026-11-22 02:00:00+00:00 | Cierre: 0.007355
Fecha: 2026-11-22 03:00:00+00:00 | Cierre: 0.007317
Fecha: 2026-11-23 01:00:00+00:00 | Cierre: 0.00765

 Barrido de máximos
Total de eventos: 60
Fecha: 2026-11-20 22:00:00+00:00 | Cierre: 0.007887
Fecha: 2026-11-21 14:00:00+00:00 | Cierre: 0.007655
Fecha: 2026-11-22 10:00:00+00:00 | Cierre: 0.00781
Fecha: 2026-11-22 17:00:00+00:00 | Cierre: 0.007781
Fecha: 2026-11-23 04:00:00+00:00 | Cierre: 0.007767

 Barrido de mínimos
Total de eventos: 103
Fecha: 2026-11-23 09:00:00+00:00 | Cierre: 0.007695

In [ ]:

# ============================================================
# CRYPTOBOT AI — BABYUSDT
# MODULO 6.2 — SMC AVANZADO
# Swing High / Swing Low / BOS / CHOCH / Sweep / FVG
# Solo análisis histórico. No ejecuta operaciones.
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. CONFIGURACION
# ------------------------------------------------------------

CONFIG_SMC = {
    "left_bars": 3,
    "right_bars": 3,
    "fvg_atr_filter": 0.10
}

# ------------------------------------------------------------
# 2. CALCULAR SWINGS CONFIRMADOS
# ------------------------------------------------------------

def detectar_swings(df, left=3, right=3):

    datos = df.copy()

    ventana = left + right + 1

    maximos = datos["high"].rolling(
        ventana, center=True
    ).max()

    minimos = datos["low"].rolling(
        ventana, center=True
    ).min()

    # Pivotes detectados en su vela original.
    swing_high_original = (
        datos["high"] == maximos
    )

    swing_low_original = (
        datos["low"] == minimos
    )

    # Se desplazan hasta la vela en que quedan confirmados.
    datos["SWING_HIGH"] = (
        swing_high_original.shift(right)
        .fillna(False)
        .astype(bool)
    )

    datos["SWING_LOW"] = (
        swing_low_original.shift(right)
        .fillna(False)
        .astype(bool)
    )

    datos["SWING_HIGH_PRICE"] = np.where(
        datos["SWING_HIGH"],
        datos["high"].shift(right),
        np.nan
    )

    datos["SWING_LOW_PRICE"] = np.where(
        datos["SWING_LOW"],
        datos["low"].shift(right),
        np.nan
    )

    return datos


# ------------------------------------------------------------
# 3. CALCULAR BOS, CHOCH Y BARRIDOS
# ------------------------------------------------------------

def detectar_estructura_avanzada(df):

    datos = df.copy()

    ultimo_maximo = (
        datos["SWING_HIGH_PRICE"].ffill().shift(1)
    )

    ultimo_minimo = (
        datos["SWING_LOW_PRICE"].ffill().shift(1)
    )

    datos["ULTIMO_SWING_HIGH"] = ultimo_maximo
    datos["ULTIMO_SWING_LOW"] = ultimo_minimo

    datos["BOS_ALCISTA"] = False
    datos["BOS_BAJISTA"] = False
    datos["CHOCH_ALCISTA"] = False
    datos["CHOCH_BAJISTA"] = False

    datos["BARRIDO_MAXIMOS"] = False
    datos["BARRIDO_MINIMOS"] = False

    direccion = 0

    for i in range(len(datos)):

        cierre = datos["close"].iloc[i]
        maximo = ultimo_maximo.iloc[i]
        minimo = ultimo_minimo.iloc[i]

        high = datos["high"].iloc[i]
        low = datos["low"].iloc[i]

        if pd.notna(maximo):

            if high > maximo and cierre < maximo:
                datos.iloc[
                    i, datos.columns.get_loc("BARRIDO_MAXIMOS")
                ] = True

        if pd.notna(minimo):

            if low < minimo and cierre > minimo:
                datos.iloc[
                    i, datos.columns.get_loc("BARRIDO_MINIMOS")
                ] = True

        ruptura_alcista = (
            pd.notna(maximo) and cierre > maximo
        )

        ruptura_bajista = (
            pd.notna(minimo) and cierre < minimo
        )

        if ruptura_alcista:

            if direccion == -1:
                datos.iloc[
                    i, datos.columns.get_loc("CHOCH_ALCISTA")
                ] = True
            else:
                datos.iloc[
                    i, datos.columns.get_loc("BOS_ALCISTA")
                ] = True

            direccion = 1

        elif ruptura_bajista:

            if direccion == 1:
                datos.iloc[
                    i, datos.columns.get_loc("CHOCH_BAJISTA")
                ] = True
            else:
                datos.iloc[
                    i, datos.columns.get_loc("BOS_BAJISTA")
                ] = True

            direccion = -1

    datos["DIRECCION_ESTRUCTURA"] = direccion

    return datos


# ------------------------------------------------------------
# 4. DETECTAR FVG CON FILTRO ATR
# ------------------------------------------------------------

def detectar_fvg_avanzado(df, filtro_atr=0.10):

    datos = df.copy()

    if "ATR14" not in datos.columns:

        anterior = datos["close"].shift(1)

        tr = pd.concat([
            datos["high"] - datos["low"],
            (datos["high"] - anterior).abs(),
            (datos["low"] - anterior).abs()
        ], axis=1).max(axis=1)

        datos["ATR14"] = tr.ewm(
            alpha=1/14,
            adjust=False
        ).mean()

    gap_alcista = (
        datos["low"] - datos["high"].shift(2)
    )

    gap_bajista = (
        datos["low"].shift(2) - datos["high"]
    )

    datos["FVG_ALCISTA"] = (
        (gap_alcista > datos["ATR14"] * filtro_atr)
        & gap_alcista.notna()
    )

    datos["FVG_BAJISTA"] = (
        (gap_bajista > datos["ATR14"] * filtro_atr)
        & gap_bajista.notna()
    )

    datos["TAMANO_FVG_ALCISTA"] = np.where(
        datos["FVG_ALCISTA"], gap_alcista, np.nan
    )

    datos["TAMANO_FVG_BAJISTA"] = np.where(
        datos["FVG_BAJISTA"], gap_bajista, np.nan
    )

    return datos


# ------------------------------------------------------------
# 5. PROCESAR CADA TEMPORALIDAD
# ------------------------------------------------------------

def procesar_smc_avanzado(df):

    resultado = detectar_swings(
        df,
        CONFIG_SMC["left_bars"],
        CONFIG_SMC["right_bars"]
    )

    resultado = detectar_estructura_avanzada(
        resultado
    )

    resultado = detectar_fvg_avanzado(
        resultado,
        CONFIG_SMC["fvg_atr_filter"]
    )

    return resultado


baby_1h_smc_avanzado = procesar_smc_avanzado(baby_1h)
baby_15m_smc_avanzado = procesar_smc_avanzado(baby_15m)
baby_5m_smc_avanzado = procesar_smc_avanzado(baby_5m)


# ------------------------------------------------------------
# 6. RESUMEN DE EVENTOS
# ------------------------------------------------------------

def resumen_smc(nombre, df):

    columnas = [
        "SWING_HIGH",
        "SWING_LOW",
        "BOS_ALCISTA",
        "BOS_BAJISTA",
        "CHOCH_ALCISTA",
        "CHOCH_BAJISTA",
        "BARRIDO_MAXIMOS",
        "BARRIDO_MINIMOS",
        "FVG_ALCISTA",
        "FVG_BAJISTA"
    ]

    print("\n====================================")
    print("SMC AVANZADO:", nombre)
    print("====================================")

    for columna in columnas:

        total = int(df[columna].sum())

        print(f"{columna}: {total}")

    print("\nULTIMA VELA:")

    ultima = df.iloc[-1]

    print("Fecha:", ultima["open_time"])
    print("Cierre:", ultima["close"])
    print("EMA21:", ultima.get("EMA21", "No disponible"))
    print("EMA50:", ultima.get("EMA50", "No disponible"))
    print("RSI14:", ultima.get("RSI14", "No disponible"))
    print("ATR14:", ultima.get("ATR14", "No disponible"))

    print("\nULTIMO SWING HIGH:",
          ultima["ULTIMO_SWING_HIGH"])

    print("ULTIMO SWING LOW:",
          ultima["ULTIMO_SWING_LOW"])


# ------------------------------------------------------------
# 7. MOSTRAR RESULTADOS
# ------------------------------------------------------------

resumen_smc("1H", baby_1h_smc_avanzado)
resumen_smc("15M", baby_15m_smc_avanzado)
resumen_smc("5M", baby_5m_smc_avanzado)

print("\n====================================")
print("MODULO 6.2 COMPLETADO")
print("====================================")
print("Analisis historico terminado.")
print("No se ejecutaron operaciones.")


/tmp/ipykernel_3060/2703008305.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_3060/2703008305.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_3060/2703008305.py:51: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_3060/2703008305.py:57: Futur


SMC AVANZADO: 1H
SWING_HIGH: 79
SWING_LOW: 73
BOS_ALCISTA: 55
BOS_BAJISTA: 149
CHOCH_ALCISTA: 12
CHOCH_BAJISTA: 12
BARRIDO_MAXIMOS: 44
BARRIDO_MINIMOS: 55
FVG_ALCISTA: 37
FVG_BAJISTA: 53

ULTIMA VELA:
Fecha: 2026-11-23 23:00:00+00:00
Cierre: 0.008481
EMA21: No disponible
EMA50: No disponible
RSI14: No disponible
ATR14: 0.00021385970110428783

ULTIMO SWING HIGH: 0.00885
ULTIMO SWING LOW: 0.007837

SMC AVANZADO: 15M
SWING_HIGH: 349
SWING_LOW: 358
BOS_ALCISTA: 369
BOS_BAJISTA: 538
CHOCH_ALCISTA: 51
CHOCH_BAJISTA: 52
BARRIDO_MAXIMOS: 135
BARRIDO_MINIMOS: 175
FVG_ALCISTA: 210
FVG_BAJISTA: 256

ULTIMA VELA:
Fecha: 2026-11-23 23:45:00+00:00
Cierre: 0.008481
EMA21: No disponible
EMA50: No disponible
RSI14: No disponible
ATR14: 0.00013016118175708863

ULTIMO SWING HIGH: 0.008536
ULTIMO SWING LOW: 0.008263

SMC AVANZADO: 5M
SWING_HIGH: 1279
SWING_LOW: 1306
BOS_ALCISTA: 1160
BOS_BAJISTA: 1445
CHOCH_ALCISTA: 156
CHOCH_BAJISTA: 155
BARRIDO_MAXIMOS: 370
BARRIDO_MINIMOS: 439
FVG_ALCISTA: 691
FVG_BAJ

In [ ]:

# ============================================================
# CRYPTOBOT AI — BABYUSDT
# MODULO 6.2.1 — CORRECCION Y FILTRADO SMC
# Evita BOS repetidos y corrige avisos booleanos
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. CONFIGURACION
# ------------------------------------------------------------

CONFIG_SMC_2 = {
    "left_bars": 3,
    "right_bars": 3,
    "fvg_atr_filter": 0.25
}

# ------------------------------------------------------------
# 2. SWINGS CONFIRMADOS
# ------------------------------------------------------------

def detectar_swings_v2(df, left=3, right=3):

    datos = df.copy()

    ventana = left + right + 1

    maximos = datos["high"].rolling(
        ventana, center=True
    ).max()

    minimos = datos["low"].rolling(
        ventana, center=True
    ).min()

    swing_high = (
        (datos["high"] == maximos)
        .shift(right)
        .astype("boolean")
        .fillna(False)
        .to_numpy(dtype=bool)
    )

    swing_low = (
        (datos["low"] == minimos)
        .shift(right)
        .astype("boolean")
        .fillna(False)
        .to_numpy(dtype=bool)
    )

    datos["SWING_HIGH"] = swing_high
    datos["SWING_LOW"] = swing_low

    datos["SWING_HIGH_PRICE"] = np.where(
        swing_high,
        datos["high"].shift(right),
        np.nan
    )

    datos["SWING_LOW_PRICE"] = np.where(
        swing_low,
        datos["low"].shift(right),
        np.nan
    )

    return datos


# ------------------------------------------------------------
# 3. BOS / CHOCH / SWEEPS SIN RUPTURAS REPETIDAS
# ------------------------------------------------------------

def detectar_estructura_v2(df):

    datos = df.copy()

    maximo = (
        datos["SWING_HIGH_PRICE"].ffill().shift(1)
    )

    minimo = (
        datos["SWING_LOW_PRICE"].ffill().shift(1)
    )

    cierre = datos["close"]
    cierre_anterior = cierre.shift(1)

    # Cruce real del nivel estructural
    ruptura_alcista = (
        (cierre_anterior <= maximo)
        & (cierre > maximo)
        & maximo.notna()
    )

    ruptura_bajista = (
        (cierre_anterior >= minimo)
        & (cierre < minimo)
        & minimo.notna()
    )

    # Barridos de liquidez
    barrido_maximos = (
        (datos["high"] > maximo)
        & (cierre < maximo)
        & maximo.notna()
    )

    barrido_minimos = (
        (datos["low"] < minimo)
        & (cierre > minimo)
        & minimo.notna()
    )

    # Clasificación BOS / CHOCH
    bos_alcista = np.zeros(len(datos), dtype=bool)
    bos_bajista = np.zeros(len(datos), dtype=bool)
    choch_alcista = np.zeros(len(datos), dtype=bool)
    choch_bajista = np.zeros(len(datos), dtype=bool)

    direccion = 0

    for i in range(len(datos)):

        if ruptura_alcista.iloc[i]:

            if direccion == -1:
                choch_alcista[i] = True
            else:
                bos_alcista[i] = True

            direccion = 1

        elif ruptura_bajista.iloc[i]:

            if direccion == 1:
                choch_bajista[i] = True
            else:
                bos_bajista[i] = True

            direccion = -1

    datos["ULTIMO_SWING_HIGH"] = maximo
    datos["ULTIMO_SWING_LOW"] = minimo

    datos["BOS_ALCISTA"] = bos_alcista
    datos["BOS_BAJISTA"] = bos_bajista
    datos["CHOCH_ALCISTA"] = choch_alcista
    datos["CHOCH_BAJISTA"] = choch_bajista

    datos["BARRIDO_MAXIMOS"] = barrido_maximos.to_numpy()
    datos["BARRIDO_MINIMOS"] = barrido_minimos.to_numpy()

    datos["DIRECCION_ESTRUCTURA"] = direccion

    return datos


# ------------------------------------------------------------
# 4. FVG CON FILTRO ATR MAS EXIGENTE
# ------------------------------------------------------------

def detectar_fvg_v2(df, filtro_atr=0.25):

    datos = df.copy()

    if "ATR14" not in datos.columns:

        anterior = datos["close"].shift(1)

        tr = pd.concat([
            datos["high"] - datos["low"],
            (datos["high"] - anterior).abs(),
            (datos["low"] - anterior).abs()
        ], axis=1).max(axis=1)

        datos["ATR14"] = tr.ewm(
            alpha=1/14,
            adjust=False
        ).mean()

    gap_alcista = (
        datos["low"] - datos["high"].shift(2)
    )

    gap_bajista = (
        datos["low"].shift(2) - datos["high"]
    )

    datos["FVG_ALCISTA"] = (
        (gap_alcista > datos["ATR14"] * filtro_atr)
        & gap_alcista.notna()
    )

    datos["FVG_BAJISTA"] = (
        (gap_bajista > datos["ATR14"] * filtro_atr)
        & gap_bajista.notna()
    )

    datos["TAMANO_FVG_ALCISTA"] = np.where(
        datos["FVG_ALCISTA"], gap_alcista, np.nan
    )

    datos["TAMANO_FVG_BAJISTA"] = np.where(
        datos["FVG_BAJISTA"], gap_bajista, np.nan
    )

    return datos


# ------------------------------------------------------------
# 5. INDICADORES PARA EL RESUMEN
# ------------------------------------------------------------

def agregar_indicadores_resumen(df):

    datos = df.copy()

    datos["EMA21"] = datos["close"].ewm(
        span=21, adjust=False
    ).mean()

    datos["EMA50"] = datos["close"].ewm(
        span=50, adjust=False
    ).mean()

    cambio = datos["close"].diff()

    ganancias = cambio.clip(lower=0)
    perdidas = -cambio.clip(upper=0)

    media_ganancias = ganancias.ewm(
        alpha=1/14, adjust=False
    ).mean()

    media_perdidas = perdidas.ewm(
        alpha=1/14, adjust=False
    ).mean()

    rs = media_ganancias / media_perdidas.replace(0, np.nan)

    datos["RSI14"] = 100 - (100 / (1 + rs))

    return datos


# ------------------------------------------------------------
# 6. PROCESAR TEMPORALIDADES
# ------------------------------------------------------------

def procesar_smc_v2(df):

    resultado = detectar_swings_v2(
        df,
        CONFIG_SMC_2["left_bars"],
        CONFIG_SMC_2["right_bars"]
    )

    resultado = detectar_estructura_v2(resultado)

    resultado = detectar_fvg_v2(
        resultado,
        CONFIG_SMC_2["fvg_atr_filter"]
    )

    resultado = agregar_indicadores_resumen(resultado)

    return resultado


baby_1h_smc_v2 = procesar_smc_v2(baby_1h)
baby_15m_smc_v2 = procesar_smc_v2(baby_15m)
baby_5m_smc_v2 = procesar_smc_v2(baby_5m)


# ------------------------------------------------------------
# 7. RESUMEN
# ------------------------------------------------------------

def resumen_smc_v2(nombre, df):

    columnas = [
        "SWING_HIGH",
        "SWING_LOW",
        "BOS_ALCISTA",
        "BOS_BAJISTA",
        "CHOCH_ALCISTA",
        "CHOCH_BAJISTA",
        "BARRIDO_MAXIMOS",
        "BARRIDO_MINIMOS",
        "FVG_ALCISTA",
        "FVG_BAJISTA"
    ]

    print("\n====================================")
    print("SMC CORREGIDO:", nombre)
    print("====================================")

    for columna in columnas:
        print(columna, ":", int(df[columna].sum()))

    ultima = df.iloc[-1]

    print("\nULTIMA VELA")
    print("Fecha:", ultima["open_time"])
    print("Cierre:", ultima["close"])
    print("EMA21:", round(ultima["EMA21"], 8))
    print("EMA50:", round(ultima["EMA50"], 8))
    print("RSI14:", round(ultima["RSI14"], 2))
    print("ATR14:", round(ultima["ATR14"], 8))

    print("ULTIMO SWING HIGH:",
          ultima["ULTIMO_SWING_HIGH"])

    print("ULTIMO SWING LOW:",
          ultima["ULTIMO_SWING_LOW"])


# ------------------------------------------------------------
# 8. RESULTADOS FINALES
# ------------------------------------------------------------

resumen_smc_v2("1H", baby_1h_smc_v2)
resumen_smc_v2("15M", baby_15m_smc_v2)
resumen_smc_v2("5M", baby_5m_smc_v2)

print("\n====================================")
print("MODULO 6.2.1 COMPLETADO")
print("====================================")
print("Sin operaciones ni ordenes reales.")



SMC CORREGIDO: 1H
SWING_HIGH : 79
SWING_LOW : 73
BOS_ALCISTA : 7
BOS_BAJISTA : 32
CHOCH_ALCISTA : 12
CHOCH_BAJISTA : 12
BARRIDO_MAXIMOS : 44
BARRIDO_MINIMOS : 55
FVG_ALCISTA : 23
FVG_BAJISTA : 36

ULTIMA VELA
Fecha: 2026-11-23 23:00:00+00:00
Cierre: 0.008481
EMA21: 0.0080999
EMA50: 0.00791568
RSI14: 65.76
ATR14: 0.00021386
ULTIMO SWING HIGH: 0.00885
ULTIMO SWING LOW: 0.007837

SMC CORREGIDO: 15M
SWING_HIGH : 349
SWING_LOW : 358
BOS_ALCISTA : 63
BOS_BAJISTA : 97
CHOCH_ALCISTA : 51
CHOCH_BAJISTA : 52
BARRIDO_MAXIMOS : 135
BARRIDO_MINIMOS : 175
FVG_ALCISTA : 139
FVG_BAJISTA : 169

ULTIMA VELA
Fecha: 2026-11-23 23:45:00+00:00
Cierre: 0.008481
EMA21: 0.00834908
EMA50: 0.00818132
RSI14: 60.06
ATR14: 0.00013016
ULTIMO SWING HIGH: 0.008536
ULTIMO SWING LOW: 0.008263

SMC CORREGIDO: 5M
SWING_HIGH : 1279
SWING_LOW : 1306
BOS_ALCISTA : 206
BOS_BAJISTA : 286
CHOCH_ALCISTA : 156
CHOCH_BAJISTA : 155
BARRIDO_MAXIMOS : 370
BARRIDO_MINIMOS : 439
FVG_ALCISTA : 547
FVG_BAJISTA : 651

ULTIMA VELA
Fecha: 

In [ ]:

# ============================================================
# CRYPTOBOT AI — BABYUSDT
# MODULO 6.3 — CONFIRMACION MULTITEMPORAL
# 4H / 1H / 15M / 5M
# Solo analisis. No ejecuta operaciones.
# ============================================================

# ------------------------------------------------------------
# 1. PROCESAR TAMBIEN LA TEMPORALIDAD 4H
# ------------------------------------------------------------

baby_4h_smc_v2 = procesar_smc_v2(baby_4h)


# ------------------------------------------------------------
# 2. CLASIFICAR TENDENCIA
# ------------------------------------------------------------

def clasificar_tendencia(df):

    ultima = df.iloc[-1]

    precio = ultima["close"]
    ema21 = ultima["EMA21"]
    ema50 = ultima["EMA50"]
    rsi = ultima["RSI14"]

    if precio > ema21 > ema50 and rsi > 50:
        return "ALCISTA"

    elif precio < ema21 < ema50 and rsi < 50:
        return "BAJISTA"

    else:
        return "NEUTRAL"


# ------------------------------------------------------------
# 3. BUSCAR EL EVENTO ESTRUCTURAL RECIENTE
# ------------------------------------------------------------

def ultimo_evento_estructural(df, barras=20):

    columnas = [
        "BOS_ALCISTA",
        "BOS_BAJISTA",
        "CHOCH_ALCISTA",
        "CHOCH_BAJISTA"
    ]

    recientes = df.tail(barras)

    mascara = recientes[columnas].any(axis=1)

    eventos = recientes[mascara]

    if eventos.empty:
        return "SIN EVENTO RECIENTE"

    ultima = eventos.iloc[-1]

    if ultima["CHOCH_ALCISTA"]:
        return "CHOCH ALCISTA"

    if ultima["CHOCH_BAJISTA"]:
        return "CHOCH BAJISTA"

    if ultima["BOS_ALCISTA"]:
        return "BOS ALCISTA"

    if ultima["BOS_BAJISTA"]:
        return "BOS BAJISTA"

    return "SIN EVENTO"


# ------------------------------------------------------------
# 4. ANALIZAR CADA TEMPORALIDAD
# ------------------------------------------------------------

temporalidades = {
    "4H": baby_4h_smc_v2,
    "1H": baby_1h_smc_v2,
    "15M": baby_15m_smc_v2,
    "5M": baby_5m_smc_v2
}

resultados_63 = {}

print("\n====================================")
print("CRYPTOBOT AI — ANALISIS MULTITEMPORAL")
print("====================================")

for nombre, df in temporalidades.items():

    tendencia = clasificar_tendencia(df)

    evento = ultimo_evento_estructural(df)

    ultima = df.iloc[-1]

    resultados_63[nombre] = {
        "tendencia": tendencia,
        "evento": evento,
        "precio": ultima["close"],
        "fecha": ultima["open_time"],
        "rsi": ultima["RSI14"]
    }

    print("\nTEMPORALIDAD:", nombre)
    print("Tendencia:", tendencia)
    print("Evento estructural:", evento)
    print("Precio histórico:", ultima["close"])
    print("RSI:", round(ultima["RSI14"], 2))
    print("Última vela:", ultima["open_time"])


# ------------------------------------------------------------
# 5. EVALUAR ALINEACION
# ------------------------------------------------------------

t4 = resultados_63["4H"]["tendencia"]
t1 = resultados_63["1H"]["tendencia"]
t15 = resultados_63["15M"]["tendencia"]
t5 = resultados_63["5M"]["tendencia"]

print("\n====================================")
print("EVALUACION DE ALINEACION")
print("====================================")

if all(t == "ALCISTA" for t in [t4, t1, t15, t5]):

    contexto = "ALINEACION ALCISTA"

elif all(t == "BAJISTA" for t in [t4, t1, t15, t5]):

    contexto = "ALINEACION BAJISTA"

elif t4 == "ALCISTA" and t1 == "ALCISTA":

    contexto = "CONTEXTO PRINCIPAL ALCISTA — ESPERAR CONFIRMACION"

elif t4 == "BAJISTA" and t1 == "BAJISTA":

    contexto = "CONTEXTO PRINCIPAL BAJISTA — ESPERAR CONFIRMACION"

else:

    contexto = "SIN ALINEACION — ESPERAR"


print("4H:", t4)
print("1H:", t1)
print("15M:", t15)
print("5M:", t5)

print("\nCONTEXTO GENERAL:", contexto)

print("\n====================================")
print("MODULO 6.3 COMPLETADO")
print("====================================")
print("No se generaron entradas.")
print("No se ejecutaron operaciones.")



CRYPTOBOT AI — ANALISIS MULTITEMPORAL

TEMPORALIDAD: 4H
Tendencia: NEUTRAL
Evento estructural: CHOCH ALCISTA
Precio histórico: 0.008481
RSI: 65.95
Última vela: 2026-11-23 20:00:00+00:00

TEMPORALIDAD: 1H
Tendencia: ALCISTA
Evento estructural: BOS ALCISTA
Precio histórico: 0.008481
RSI: 65.76
Última vela: 2026-11-23 23:00:00+00:00

TEMPORALIDAD: 15M
Tendencia: ALCISTA
Evento estructural: CHOCH BAJISTA
Precio histórico: 0.008481
RSI: 60.06
Última vela: 2026-11-23 23:45:00+00:00

TEMPORALIDAD: 5M
Tendencia: ALCISTA
Evento estructural: SIN EVENTO RECIENTE
Precio histórico: 0.008481
RSI: 61.17
Última vela: 2026-11-23 23:55:00+00:00

EVALUACION DE ALINEACION
4H: NEUTRAL
1H: ALCISTA
15M: ALCISTA
5M: ALCISTA

CONTEXTO GENERAL: SIN ALINEACION — ESPERAR

MODULO 6.3 COMPLETADO
No se generaron entradas.
No se ejecutaron operaciones.
